**Photos Library Notebook — VS Code navigation and execution rules**

This notebook is deliberately **not** a Run-All notebook.

- Every executable operation is placed in its own `##` Markdown section.
- In VS Code Outline, click a section to jump to it.
- Use **Run Cells in Section** on that heading; because the next heading is also `##`, it executes exactly the single code cell immediately below it.
- For a long code cell, collapse the cell input using the upper half of the blue gutter after navigating. Its output then sits directly below the heading/collapsed cell.
- `DELETE-02 — Post-Delete Audit` must use the exact manifest that was executed. Do not rerun `DELETE-01` first, because `DELETE-01` overwrites the Latest manifest.
- `REPAIR-03` is high-risk because its output is consumed by a macOS app that modifies Photos.

**Common duplicate-cleanup order:** `SETUP-01` → `SETUP-02` → `REPORT-02` → `DELETE-01` → macOS app dry run/write → fresh `SETUP-02` rebuild → `DELETE-02` → `REPORT-02` → `DELETE-01` (expect zero strict rows for the completed target).


---

**Companion file:** `Photos_Library_Repair_Notebook_Scenario_Runbook_2026-06-22.md`

Use the companion runbook to choose one scenario at a time. Do not treat every repair section as a sequence that should always be executed.


## SETUP-01 — Imports and Settings

**Purpose:** Load imports, helper modules, paths, and rebuild flags.  
**Execution:** Run before every scenario.  
**Risk:** Configuration only; verify all force-rebuild and target-folder settings before continuing.


In [ ]:
# ============================================================
# Imports and Settings
# ============================================================

from pathlib import Path
from datetime import datetime
import importlib

import osxphotos

# Reload helper module without restarting the Jupyter kernel.
#
# This is important because:
# - In Jupyter, `from explorephotoslibrary import *` does NOT automatically
#   pick up edits made to explorephotoslibrary.py after the first import.
# - Restarting the kernel would lose the current notebook state.
# - Reloading the module here lets later cells use the updated functions.
import explorephotoslibrary as _explorephotoslibrary
importlib.reload(_explorephotoslibrary)

from explorephotoslibrary import *


USE_INVENTORY_CACHE = False


# Rebuild only the libraries listed here.
#
# Normal use:

# FORCE_REBUILD_INVENTORY_KEYS = set()


# Rebuild Current Default only:
# FORCE_REBUILD_INVENTORY_KEYS = {"current_default"}
#
# Rebuild both:
# FORCE_REBUILD_INVENTORY_KEYS = {
#     "backup_20250317",
#     "current_default",
# }

FORCE_REBUILD_INVENTORY_KEYS = {
    "current_default",
    "backup_20250317",
}


# Path-selection policy:
#
# False:
# - Reuse each library path saved in:
#     data/local_config/test2_library_paths.json
#
# True:
# - Reselect the path only for libraries listed in
#   FORCE_REBUILD_INVENTORY_KEYS.
# - Libraries not listed in FORCE_REBUILD_INVENTORY_KEYS continue
#   using their saved paths.
#
# Current situation:
# - Current Default Photos Library was moved.
# - Backup Photos Library did not move.
# Therefore:
#     FORCE_REBUILD_INVENTORY_KEYS = {"current_default"}
#     FORCE_RESELECT_LIBRARY_PATHS = True
FORCE_RESELECT_LIBRARY_PATHS = False


TEST2_LIBRARY_PROMPTS = {
    "backup_20250317": (
        "Select BACKUP Photos Library: backup_20250317"
    ),
    "current_default": (
        "Select CURRENT default Photos Library: current_default"
    ),
}


TEST2_DEFAULT_INITIAL_DIRS = {
    "backup_20250317": "/Volumes",
    "current_default": str(Path.home() / "Pictures"),
}


# Short visible confirmation that this settings cell really ran.
{
    "use_inventory_cache": USE_INVENTORY_CACHE,
    "force_rebuild_inventory_keys": sorted(
        FORCE_REBUILD_INVENTORY_KEYS
    ),
    "force_reselect_library_paths": (
        FORCE_RESELECT_LIBRARY_PATHS
    ),
    "libraries_that_will_be_reselected": (
        sorted(FORCE_REBUILD_INVENTORY_KEYS)
        if FORCE_RESELECT_LIBRARY_PATHS
        else []
    ),
}

{'use_inventory_cache': False,
 'force_rebuild_inventory_keys': ['backup_20250317', 'current_default'],
 'force_reselect_library_paths': False,
 'libraries_that_will_be_reselected': []}

## SETUP-02 — Load or Build Inventories

**Purpose:** Load cached inventories or rebuild the selected libraries.  
**Requires:** `SETUP-01`.  
**Risk:** Read-only to Photos, but rebuilding can take a long time.


In [2]:
# =====================================================================================
# Load or build inventories — unique ID generation / validation included
# =====================================================================================

TEST2_LIBRARY_HISTORY_PATH = Path("data/local_config/test2_library_paths.json")

def get_test2_library_path(library_key):
    library_history = load_json_file(
        TEST2_LIBRARY_HISTORY_PATH,
        default={},
    ) or {}

    saved_library_path = library_history.get(library_key)

    # FORCE_RESELECT_LIBRARY_PATHS is one global True/False switch.
    #
    # When True, reselect only libraries also listed in
    # FORCE_REBUILD_INVENTORY_KEYS.
    should_force_reselect = (
        FORCE_RESELECT_LIBRARY_PATHS
        and library_key in FORCE_REBUILD_INVENTORY_KEYS
    )

    if (
        saved_library_path
        and Path(saved_library_path).exists()
        and not should_force_reselect
    ):
        library_path = Path(saved_library_path)

        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(f"{library_key} library path:", library_path)
        print()

        return library_path

    if saved_library_path:
        initial_dir = Path(saved_library_path).parent
    else:
        initial_dir = Path(
            TEST2_DEFAULT_INITIAL_DIRS.get(
                library_key,
                "/Volumes",
            )
        )

    prompt = TEST2_LIBRARY_PROMPTS.get(
        library_key,
        f"Select Photos Library for: {library_key}",
    )

    print("=" * 80)

    if should_force_reselect:
        print(f"Force reselect Photos Library path for: {library_key}")
    else:
        print(prompt)

    print("=" * 80)

    library_path = Path(
        choose_photos_library_path(
            initial_dir=initial_dir,
            prompt=prompt,
        )
    )

    library_history[library_key] = str(library_path)
    library_history[f"{library_key}_selected_at"] = (
        datetime.now().isoformat()
    )

    save_json_file(
        TEST2_LIBRARY_HISTORY_PATH,
        library_history,
    )

    print(f"{library_key} library path:", library_path)
    print()

    return library_path

def print_section2_identity_summary(inventory, label):
    key_to_count = {}
    assets_without_unique_id = 0

    for asset in inventory.get("assets") or []:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            assets_without_unique_id += 1
            continue

        key_to_count[unique_id] = key_to_count.get(unique_id, 0) + 1

    duplicate_group_count = sum(
        1
        for count in key_to_count.values()
        if count > 1
    )

    duplicate_asset_count = sum(
        count
        for count in key_to_count.values()
        if count > 1
    )

    is_ok = (
        assets_without_unique_id == 0
        and duplicate_group_count == 0
    )

    print()
    print(f"{label} identity summary")
    print("-" * 80)
    print("total asset count:", len(inventory.get("assets") or []))
    print("generated unique ID count:", len(key_to_count))
    print("assets without unique ID:", assets_without_unique_id)
    print("duplicate unique ID group count:", duplicate_group_count)
    print("duplicate asset count:", duplicate_asset_count)
    print("photo_library_asset_unique_id status:", "OK" if is_ok else "FAILED")

    if not is_ok:
        raise RuntimeError(f"{label} inventory identity validation failed.")


def load_or_build_inventory(library_key):
    library_path = get_test2_library_path(library_key)
    should_rebuild_inventory = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild_inventory:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            inventory = load_inventory_cache(library_key)
            print_section2_identity_summary(inventory, library_key)
            return inventory
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    if should_rebuild_inventory:
        print("=" * 80)
        print(f"Force rebuild inventory: {library_key}")
        print("=" * 80)
    else:
        print("=" * 80)
        print(f"Build inventory: {library_key}")
        print("=" * 80)

    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)
    print_section2_identity_summary(inventory, library_key)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Use saved Photos Library path for: backup_20250317
backup_20250317 library path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary

Force rebuild inventory: backup_20250317
backup_20250317 osx asset count: 71572
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000

backup_20250317 inventory summary
--------------------------------------------------------------------------------
inventory assets: 71572
inventory albums: 5110
inventory folders: 35
special assets:
  PATH_MISSING: 0
  PATH_NONE: 0
  SYNDICATED_NO_NORMAL_ORIGINAL: 0
  UNKNOWN_PATH: 0
movies: 6224
hidden: 0
favorites: 701
descriptions: 728
keywords: 23747

backup_20250317 identity summary
-------------------------------------------------------

## REPORT-01 — Backup-Centric Restoration Dashboard

**Purpose:** Prove whether all Backup-required information has been restored into Current.

**Asset-level preservation checks:**

1. **Backup Asset Dates Different from Current**  
   Compare the full Backup and Current date in Asia/Taipei local time through whole-second precision: `YYYY-MM-DD HH:MM:SS`.  
   The year is compared. Microseconds are ignored.

2. **Backup Asset Descriptions Missing or Different in Current**  
   When Backup has a description, Current must retain the same description.

3. **Backup Keywords Missing from Current**  
   Every Backup keyword must still be present in Current. Current may contain additional keywords.

4. **Backup GPS Present but Missing from Current**  
   When Backup has both latitude and longitude, Current must also have both.

5. **Backup and Current GPS Coordinates Differ**  
   When both have GPS, compare latitude and longitude rounded to four decimal places.

6. **Backup Media or Adjustment Metadata Different from Current**  
   Compare media type, dimensions, and adjustment-related fields.

**Structural preservation checks:**

- Backup Assets Missing from Current  
- Backup Folder Paths Missing from Current  
- Backup Album Paths Missing from Current  
- Backup Album Members Missing from Current Albums at Matching Album Paths  
- Backup Album Paths Associated with Multiple Albums  
- Current Album Paths Associated with Multiple Albums  

**Completion criterion:** All dashboard counters must be `0`, and `all_backup_required_information_present_in_current` must be `YES`.

**Requires:** `SETUP-02`.  
**Risk:** Read-only report generation.


In [ ]:
# ============================================================
# REPORT-01 — Backup-Centric Restoration Dashboard
# ============================================================

folder_album_membership_report = write_folder_album_membership_comparison_report(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
    report_root=Path("reports/test2_folder_album_membership_comparison"),
    label="whole_library",
    target_root_folder_path=None,
    include_ok_rows=False,
)

{
    "folder_album_membership_report": folder_album_membership_report,
}

## REPORT-01A — Whole-Second Date-Difference Review List

**Purpose:** List matched Backup/Current assets whose full Asia/Taipei date differs after microseconds are ignored.

**Requires:** `SETUP-02`.

**Risk:** Read-only. Writes a TSV under `reports/test2_asset_date_difference_review/`.

**Interpretation:** This is a review list for intentional or accidental date changes. It does not change either Photos Library.


In [ ]:
# ============================================================
# REPORT-01 — Export the 84 Date-Difference Review List
#
# Read-only. This does not modify either Photos Library.
# Requires: SETUP-02 already run, so inventory_backup and
# inventory_current exist in the notebook kernel.
# ============================================================

import csv
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

TAIPEI = ZoneInfo("Asia/Taipei")


def _date_to_taipei_seconds(value):
    """Full date including year, normalized to Taipei time and whole seconds."""
    if value is None:
        return None

    dt = value if isinstance(value, datetime) else datetime.fromisoformat(str(value))

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=TAIPEI)
    else:
        dt = dt.astimezone(TAIPEI)

    return dt.strftime("%Y-%m-%d %H:%M:%S")


def _normalized_asset_key(asset):
    """
    Match the project's cross-library identity behavior:
    normalize only original-filename extension case inside the existing
    photo_library_asset_unique_id tuple.
    """
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        raise RuntimeError(
            "Asset has no photo_library_asset_unique_id:\n"
            f"{asset.get('original_filename')!r}"
        )

    unique_id = tuple(unique_id)

    if len(unique_id) < 5:
        return unique_id

    filename = unique_id[0]

    if filename and "." in str(filename):
        stem, extension = str(filename).rsplit(".", 1)
        filename = f"{stem}.{extension.lower()}"

    return (
        filename,
        unique_id[1],
        unique_id[2],
        unique_id[3],
        unique_id[4],
    )


def _index_assets_by_key(inventory):
    index = {}

    for asset in inventory["assets"]:
        key = _normalized_asset_key(asset)

        if key in index:
            raise RuntimeError(
                "Duplicate normalized photo_library_asset_unique_id:\n"
                f"{key!r}"
            )

        index[key] = asset

    return index


backup_by_key = _index_assets_by_key(inventory_backup)
current_by_key = _index_assets_by_key(inventory_current)

rows = []

for asset_key in sorted(set(backup_by_key) & set(current_by_key), key=repr):
    backup_asset = backup_by_key[asset_key]
    current_asset = current_by_key[asset_key]

    backup_date_to_seconds = _date_to_taipei_seconds(backup_asset.get("date"))
    current_date_to_seconds = _date_to_taipei_seconds(current_asset.get("date"))

    if backup_date_to_seconds == current_date_to_seconds:
        continue

    backup_year = backup_date_to_seconds[:4] if backup_date_to_seconds else ""
    current_year = current_date_to_seconds[:4] if current_date_to_seconds else ""

    rows.append({
        "original_filename": backup_asset.get("original_filename") or "",
        "backup_date_raw": backup_asset.get("date") or "",
        "current_date_raw": current_asset.get("date") or "",
        "backup_date_taipei_to_seconds": backup_date_to_seconds or "",
        "current_date_taipei_to_seconds": current_date_to_seconds or "",
        "backup_year": backup_year,
        "current_year": current_year,
        "year_delta": (
            int(current_year) - int(backup_year)
            if backup_year and current_year
            else ""
        ),
        "asset_key": repr(asset_key),
        "backup_asset_uuid": backup_asset.get("uuid") or "",
        "current_asset_uuid": current_asset.get("uuid") or "",
        "backup_path": backup_asset.get("path") or "",
        "current_path": current_asset.get("path") or "",
    })

rows.sort(
    key=lambda row: (
        row["backup_date_taipei_to_seconds"],
        row["original_filename"],
        row["asset_key"],
    )
)

report_dir = Path("reports/test2_asset_date_difference_review")
report_dir.mkdir(parents=True, exist_ok=True)

tsv_path = report_dir / "backup_vs_current_date_differences.tsv"

fieldnames = [
    "original_filename",
    "backup_date_raw",
    "current_date_raw",
    "backup_date_taipei_to_seconds",
    "current_date_taipei_to_seconds",
    "backup_year",
    "current_year",
    "year_delta",
    "asset_key",
    "backup_asset_uuid",
    "current_asset_uuid",
    "backup_path",
    "current_path",
]

with tsv_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    writer.writerows(rows)

print("Date-difference rows:", len(rows))
print("TSV:", tsv_path)
print()
print("First 20 rows:")
for row in rows[:20]:
    print(
        f"{row['backup_date_taipei_to_seconds']}  "
        f"→ {row['current_date_taipei_to_seconds']}  "
        f"{row['original_filename']}"
    )


## REPORT-02 — Current Default Duplicate Album Titles

**Purpose:** Export the Numbers-friendly duplicate-title TSV for Current Default.  
**Requires:** `SETUP-02`.  
**Risk:** Read-only to Photos; writes a report TSV.


In [ ]:
# ============================================================
# Current Default — Duplicate Album Titles
# Numbers-friendly TSV report
#
# Duplicate title definition:
# - exact same title, OR
# - title differs only by whitespace formatting:
#   leading/trailing spaces, repeated spaces, tabs, line breaks,
#   or other Unicode whitespace
# - 2 or more different Current album UUIDs
#
# IMPORTANT:
# - punctuation is NOT ignored
# - "..." and "…" remain meaningful
# - emoji and hashtags remain meaningful
#
# READ ONLY:
# - reads inventory_current only
# - does not rebuild inventory
# - does not modify Photos Library
# ============================================================

from collections import defaultdict, Counter
from pathlib import Path
from datetime import datetime
import csv
import hashlib


def duplicate_album_safe_tsv_text(value):
    """
    Preserve the complete original text while keeping each album
    on one TSV row.

    Actual tabs and line breaks inside a title are represented
    visibly as \\t, \\r, and \\n.
    """
    return (
        str(value or "")
        .replace("\\", "\\\\")
        .replace("\t", "\\t")
        .replace("\r", "\\r")
        .replace("\n", "\\n")
    )


def duplicate_album_normalize_title_for_matching(value):
    """
    Normalize whitespace only.

    Examples treated as equivalent:

        "ABC   DEF"
        "ABC DEF"
        "ABC\\nDEF"
        " ABC DEF   "

    This does NOT remove punctuation, dots, ellipsis,
    hashtags, emoji, or other visible characters.
    """
    text = str(value or "")

    return " ".join(text.split())


def duplicate_album_title_whitespace_stats(value):
    """
    Return useful whitespace diagnostics for one raw title.
    """
    text = str(value or "")

    leading_count = (
        len(text) - len(text.lstrip())
    )

    trailing_count = (
        len(text) - len(text.rstrip())
    )

    return {
        "raw_title_length":
            len(text),

        "normalized_title_length":
            len(
                duplicate_album_normalize_title_for_matching(
                    text
                )
            ),

        "leading_whitespace_count":
            leading_count,

        "trailing_whitespace_count":
            trailing_count,

        "tab_count":
            text.count("\t"),

        "line_feed_count":
            text.count("\n"),

        "carriage_return_count":
            text.count("\r"),

        "nonbreaking_space_count":
            text.count("\u00a0"),
    }


def duplicate_album_normalize_folder_path(path):
    text = str(path or "").strip()

    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    return " / ".join(
        part.strip()
        for part in text.split("/")
        if part.strip()
    )


def duplicate_album_leaf_folder_paths(album):
    """
    Return only the deepest folder path or paths.

    Albums without a folder are reported as [ROOT_ALBUMS].
    """
    all_paths = {
        duplicate_album_normalize_folder_path(
            folder.get("path")
            or folder.get("title")
        )
        for folder in (
            album.get("folders") or {}
        ).values()
        if (
            folder.get("path")
            or folder.get("title")
        )
    }

    all_paths.discard("")

    if not all_paths:
        return ("[ROOT_ALBUMS]",)

    leaf_paths = [
        path
        for path in all_paths
        if not any(
            other != path
            and other.startswith(path + " / ")
            for other in all_paths
        )
    ]

    return tuple(sorted(leaf_paths))


def duplicate_album_membership_checksum(asset_uuids):
    payload = "\n".join(
        sorted(asset_uuids)
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:16]


# ------------------------------------------------------------
# Build exact asset membership for every Current album UUID
# ------------------------------------------------------------

members_by_album_uuid = defaultdict(set)

for asset in inventory_current.get("assets") or []:
    asset_uuid = str(
        asset.get("uuid") or ""
    )

    if not asset_uuid:
        continue

    for album_uuid in (
        asset.get("albums") or {}
    ):
        members_by_album_uuid[
            str(album_uuid)
        ].add(asset_uuid)


# ------------------------------------------------------------
# Group albums by whitespace-normalized complete title
# ------------------------------------------------------------

albums_by_normalized_title = defaultdict(list)

for inventory_key, album in (
    inventory_current.get("albums") or {}
).items():
    raw_title = album.get("title")

    if raw_title is None:
        continue

    raw_title = str(raw_title)

    normalized_title = (
        duplicate_album_normalize_title_for_matching(
            raw_title
        )
    )

    if normalized_title == "":
        continue

    album_uuid = str(
        album.get("uuid")
        or inventory_key
    )

    member_set = set(
        members_by_album_uuid.get(
            album_uuid,
            set(),
        )
    )

    whitespace_stats = (
        duplicate_album_title_whitespace_stats(
            raw_title
        )
    )

    albums_by_normalized_title[
        normalized_title
    ].append({
        "album_uuid":
            album_uuid,

        "folder_paths":
            duplicate_album_leaf_folder_paths(
                album
            ),

        "raw_title":
            raw_title,

        "normalized_title":
            normalized_title,

        "member_set":
            member_set,

        "asset_count":
            len(member_set),

        "membership_checksum":
            duplicate_album_membership_checksum(
                member_set
            ),

        **whitespace_stats,
    })


duplicate_groups = [
    (
        normalized_title,
        album_objects,
    )
    for normalized_title, album_objects
    in albums_by_normalized_title.items()
    if len({
        row["album_uuid"]
        for row in album_objects
    }) >= 2
]

duplicate_groups.sort(
    key=lambda item: item[0]
)


# ------------------------------------------------------------
# Create one Numbers-friendly row per album object
# ------------------------------------------------------------

report_rows = []

exact_title_group_count = 0
whitespace_only_title_group_count = 0

exact_membership_group_count = 0
different_membership_group_count = 0


for group_number, (
    normalized_title,
    album_objects,
) in enumerate(
    duplicate_groups,
    start=1,
):
    album_objects = sorted(
        album_objects,
        key=lambda row: (
            row["folder_paths"],
            row["album_uuid"],
        ),
    )

    total_album_objects = len(
        album_objects
    )

    extra_copy_count = (
        total_album_objects - 1
    )

    raw_title_values = {
        row["raw_title"]
        for row in album_objects
    }

    if len(raw_title_values) == 1:
        title_match_type = (
            "EXACT_TITLE"
        )

        exact_title_group_count += 1

    else:
        title_match_type = (
            "WHITESPACE_ONLY_DIFFERENCE"
        )

        whitespace_only_title_group_count += 1


    membership_sets = [
        frozenset(row["member_set"])
        for row in album_objects
    ]

    membership_set_counts = Counter(
        membership_sets
    )

    distinct_membership_sets = set(
        membership_sets
    )

    if len(distinct_membership_sets) == 1:
        group_membership_status = (
            "EXACT_SAME_MEMBERSHIP"
        )

        exact_membership_group_count += 1

    else:
        group_membership_status = (
            "DIFFERENT_MEMBERSHIP"
        )

        different_membership_group_count += 1


    for copy_number, row in enumerate(
        album_objects,
        start=1,
    ):
        membership_key = frozenset(
            row["member_set"]
        )

        report_rows.append({
            "group_number":
                group_number,

            "copy_number_in_group":
                copy_number,

            "total_album_objects_in_group":
                total_album_objects,

            "extra_copy_count_in_group":
                extra_copy_count,

            "title_match_type":
                title_match_type,

            "group_membership_status":
                group_membership_status,

            "folder_path":
                " || ".join(
                    row["folder_paths"]
                ),

            "album_title":
                row["raw_title"],

            "raw_title_length":
                row["raw_title_length"],

            "normalized_title_length":
                row["normalized_title_length"],

            "leading_whitespace_count":
                row[
                    "leading_whitespace_count"
                ],

            "trailing_whitespace_count":
                row[
                    "trailing_whitespace_count"
                ],

            "tab_count":
                row["tab_count"],

            "line_feed_count":
                row["line_feed_count"],

            "carriage_return_count":
                row[
                    "carriage_return_count"
                ],

            "nonbreaking_space_count":
                row[
                    "nonbreaking_space_count"
                ],

            "album_uuid":
                row["album_uuid"],

            "asset_count":
                row["asset_count"],

            "membership_checksum":
                row[
                    "membership_checksum"
                ],

            "copies_with_same_membership":
                membership_set_counts[
                    membership_key
                ],
        })


# ------------------------------------------------------------
# Save Numbers-friendly TSV
# ------------------------------------------------------------

run_time = datetime.now()

report_dir = (
    Path(
        "reports/"
        "test2_current_duplicate_album_titles"
    )
    / f"{run_time:%Y%m%d-%H%M%S}"
)

report_dir.mkdir(
    parents=True,
    exist_ok=False,
)

tsv_path = (
    report_dir
    / "current_duplicate_album_titles.tsv"
).resolve()


fieldnames = [
    "group_number",
    "copy_number_in_group",
    "total_album_objects_in_group",
    "extra_copy_count_in_group",

    "title_match_type",
    "group_membership_status",

    "folder_path",
    "album_title",

    "raw_title_length",
    "normalized_title_length",

    "leading_whitespace_count",
    "trailing_whitespace_count",

    "tab_count",
    "line_feed_count",
    "carriage_return_count",
    "nonbreaking_space_count",

    "album_uuid",
    "asset_count",
    "membership_checksum",
    "copies_with_same_membership",
]


with tsv_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
        delimiter="\t",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    writer.writeheader()

    for row in report_rows:
        safe_row = dict(row)

        safe_row["folder_path"] = (
            duplicate_album_safe_tsv_text(
                safe_row["folder_path"]
            )
        )

        safe_row["album_title"] = (
            duplicate_album_safe_tsv_text(
                safe_row["album_title"]
            )
        )

        writer.writerow(safe_row)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

duplicate_album_title_result = {
    "tsv_path":
        str(tsv_path),

    "duplicate_title_group_count":
        len(duplicate_groups),

    "exact_title_group_count":
        exact_title_group_count,

    "whitespace_only_title_group_count":
        whitespace_only_title_group_count,

    "exact_same_membership_group_count":
        exact_membership_group_count,

    "different_membership_group_count":
        different_membership_group_count,

    "album_object_count_in_groups":
        len(report_rows),

    "total_extra_album_objects":
        sum(
            len(album_objects) - 1
            for _, album_objects
            in duplicate_groups
        ),
}


print(
    "Duplicate title groups:",
    duplicate_album_title_result[
        "duplicate_title_group_count"
    ],
)

print(
    "Exact-title groups:",
    duplicate_album_title_result[
        "exact_title_group_count"
    ],
)

print(
    "Whitespace-only title-difference groups:",
    duplicate_album_title_result[
        "whitespace_only_title_group_count"
    ],
)

print(
    "Exact same-membership groups:",
    duplicate_album_title_result[
        "exact_same_membership_group_count"
    ],
)

print(
    "Different-membership groups:",
    duplicate_album_title_result[
        "different_membership_group_count"
    ],
)

print(
    "Total extra album objects:",
    duplicate_album_title_result[
        "total_extra_album_objects"
    ],
)

print()
print("TSV report:")
print(tsv_path)

## DELETE-01 — Export Strict Duplicate-Album Delete Manifest

**Purpose:** Use Backup as canonical ground truth and export strict delete candidates plus the executable PhotoKit manifest.  
**Requires:** Fresh or intentionally selected inventories from `SETUP-02`.  
**Risk:** Does not modify Photos, but **overwrites the Latest candidate and manifest TSV files**.


In [ ]:
# ============================================================
# Delete duplicate album shells — export strict PhotoKit manifest
#
# SAFE candidate definition:
# 1. Current has 2+ album objects whose titles are equal after
#    whitespace-only normalization.
# 2. Their complete Current asset memberships are exactly equal.
# 3. Backup has exactly one album with the same normalized title
#    and the same cross-library asset membership.
# 4. Exactly one Current album is at that Backup folder path.
# 5. The keeper's raw title exactly equals the Backup raw title.
# 6. Every delete candidate has exactly one different folder path.
#
# Excluded:
# - empty albums
# - albums with missing asset unique IDs
# - "#給資料夾置頂用"
#
# READ ONLY: writes TSV files only; does not modify Photos.
# ============================================================

from collections import defaultdict
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import json
import shutil

DELETE_DUP_INBOX = Path.home() / "Downloads" / "PhotosRepairMVP_Inbox"
DELETE_DUP_MANIFEST = DELETE_DUP_INBOX / "DeleteDuplicateAlbumsManifestLatest.tsv"
DELETE_DUP_REVIEW = DELETE_DUP_INBOX / "DeleteDuplicateAlbumsCandidatesLatest.tsv"
DELETE_DUP_ARCHIVE = DELETE_DUP_INBOX / "archive"
DELETE_DUP_EXCLUDED_TITLES = {"#給資料夾置頂用"}
# 給定刪除範圍的Root Folder
DELETE_DUP_TARGET_KEEP_FOLDER_PATH = "NSFW"

WRITE_DELETE_DUP_ARCHIVE = True


def delete_dup_normalize_title(value):
    return " ".join(str(value or "").split())


def delete_dup_normalize_path(value):
    text = str(value or "").strip().replace("\\", "/").replace(" / ", "/")
    return " / ".join(part.strip() for part in text.split("/") if part.strip())


def delete_dup_leaf_paths(album):
    paths = {
        delete_dup_normalize_path(folder.get("path") or folder.get("title"))
        for folder in (album.get("folders") or {}).values()
        if folder.get("path") or folder.get("title")
    }
    paths.discard("")
    if not paths:
        return ("[ROOT_ALBUMS]",)
    return tuple(sorted(
        path for path in paths
        if not any(other != path and other.startswith(path + " / ") for other in paths)
    ))


def delete_dup_asset_identity(asset):
    unique_id = asset.get("photo_library_asset_unique_id")
    if unique_id is None:
        return None
    return json.dumps(unique_id, ensure_ascii=False, sort_keys=True, default=str)


def delete_dup_checksum(values):
    payload = "\n".join(sorted(values))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def delete_dup_tsv(value):
    return str(value if value is not None else "-").replace("\t", " ").replace("\r", " ").replace("\n", " ")


def delete_dup_album_records(inventory):
    cross_members = defaultdict(set)
    local_uuid_members = defaultdict(set)
    missing_identity_albums = set()

    for asset in inventory.get("assets") or []:
        identity = delete_dup_asset_identity(asset)
        local_uuid = str(asset.get("uuid") or "")
        for album_uuid in (asset.get("albums") or {}):
            album_uuid = str(album_uuid)
            if identity is None:
                missing_identity_albums.add(album_uuid)
            else:
                cross_members[album_uuid].add(identity)
            if local_uuid:
                local_uuid_members[album_uuid].add(local_uuid)

    records = []
    for inventory_key, album in (inventory.get("albums") or {}).items():
        raw_title = str(album.get("title") or "")
        normalized_title = delete_dup_normalize_title(raw_title)
        if not normalized_title:
            continue

        album_uuid = str(album.get("uuid") or inventory_key)
        member_set = frozenset(cross_members.get(album_uuid, set()))
        local_uuid_set = frozenset(local_uuid_members.get(album_uuid, set()))

        records.append({
            "album_uuid": album_uuid,
            "raw_title": raw_title,
            "normalized_title": normalized_title,
            "folder_paths": delete_dup_leaf_paths(album),
            "member_set": member_set,
            "local_uuid_set": local_uuid_set,
            "asset_count": len(local_uuid_set),
            "membership_checksum": delete_dup_checksum(local_uuid_set),
            "has_missing_asset_identity": album_uuid in missing_identity_albums,
        })

    return records


current_records = delete_dup_album_records(inventory_current)
backup_records = delete_dup_album_records(inventory_backup)

backup_by_identity = defaultdict(list)
for row in backup_records:
    backup_by_identity[(row["normalized_title"], row["member_set"])].append(row)

current_by_identity = defaultdict(list)
for row in current_records:
    current_by_identity[(row["normalized_title"], row["member_set"])].append(row)

review_rows = []
manifest_rows = []
candidate_group_number = 0

for identity_key, current_group in sorted(
    current_by_identity.items(),
    key=lambda item: item[0][0],
):
    normalized_title, member_set = identity_key
    if len(current_group) < 2:
        continue

    candidate_group_number += 1
    backup_group = backup_by_identity.get(identity_key, [])
    raw_titles = {row["raw_title"] for row in current_group}
    title_match_type = (
        "EXACT_TITLE" if len(raw_titles) == 1
        else "WHITESPACE_ONLY_DIFFERENCE"
    )

    decision = "SKIP"
    reason = ""
    keeper = None
    delete_rows = []

    if normalized_title in DELETE_DUP_EXCLUDED_TITLES:
        reason = "EXCLUDED_TITLE"
    elif not member_set:
        reason = "EMPTY_MEMBERSHIP"
    elif any(row["has_missing_asset_identity"] for row in current_group):
        reason = "CURRENT_ASSET_IDENTITY_MISSING"
    elif len({row["local_uuid_set"] for row in current_group}) != 1:
        reason = "CURRENT_LOCAL_MEMBERSHIP_MISMATCH"
    elif len(backup_group) != 1:
        reason = f"BACKUP_MATCH_COUNT_{len(backup_group)}"
    else:
        backup_album = backup_group[0]

        if backup_album["has_missing_asset_identity"]:
            reason = "BACKUP_ASSET_IDENTITY_MISSING"
        elif len(backup_album["folder_paths"]) != 1:
            reason = "BACKUP_PATH_AMBIGUOUS"
        else:
            canonical_path = backup_album["folder_paths"][0]
            keepers = [
                row for row in current_group
                if row["folder_paths"] == (canonical_path,)
                and row["raw_title"] == backup_album["raw_title"]
            ]

            if len(keepers) != 1:
                reason = f"CURRENT_KEEPER_COUNT_{len(keepers)}"
            else:
                keeper = keepers[0]
                delete_rows = [
                    row for row in current_group
                    if row["album_uuid"] != keeper["album_uuid"]
                ]

                if not delete_rows:
                    reason = "NO_EXTRA_OBJECT"
                elif any(len(row["folder_paths"]) != 1 for row in delete_rows):
                    reason = "DELETE_PATH_AMBIGUOUS"
                elif any(row["folder_paths"][0] == canonical_path for row in delete_rows):
                    reason = "EXTRA_OBJECT_AT_CANONICAL_PATH"
                else:
                    decision = "READY_TO_DELETE"
                    reason = "STRICT_BACKUP_CANONICAL_MATCH"

    canonical_path = (
        backup_group[0]["folder_paths"][0]
        if len(backup_group) == 1 and len(backup_group[0]["folder_paths"]) == 1
        else "-"
    )
    
    if (
        decision == "READY_TO_DELETE"
        and canonical_path != DELETE_DUP_TARGET_KEEP_FOLDER_PATH
    ):
        decision = "SKIP"
        reason = "OUTSIDE_TARGET_KEEP_FOLDER_PATH"
        keeper = None
        delete_rows = []

    for row in current_group:
        role = (
            "KEEP" if keeper and row["album_uuid"] == keeper["album_uuid"]
            else "DELETE" if decision == "READY_TO_DELETE"
            else "REVIEW"
        )

        review_rows.append({
            "candidate_group_number": candidate_group_number,
            "decision": decision,
            "reason": reason,
            "role": role,
            "title_match_type": title_match_type,
            "folder_path": " || ".join(row["folder_paths"]),
            "backup_canonical_path": canonical_path,
            "album_title": row["raw_title"],
            "album_uuid": row["album_uuid"],
            "asset_count": row["asset_count"],
            "membership_checksum": row["membership_checksum"],
            "current_group_object_count": len(current_group),
            "backup_matching_album_count": len(backup_group),
        })

    if decision == "READY_TO_DELETE":
        for row in delete_rows:
            manifest_rows.append({
                "operation": "delete_duplicate_album",
                "delete_album_uuid": row["album_uuid"],
                "keep_album_uuid": keeper["album_uuid"],
                "delete_folder_path": row["folder_paths"][0],
                "keep_folder_path": canonical_path,
                "album_title": row["raw_title"],
                "normalized_title": normalized_title,
                "asset_count": row["asset_count"],
                "membership_checksum": row["membership_checksum"],
                "title_match_type": title_match_type,
                "backup_canonical_path": canonical_path,
                "candidate_group_number": candidate_group_number,
                "status": "READY_TO_DELETE",
            })


DELETE_DUP_INBOX.mkdir(parents=True, exist_ok=True)
DELETE_DUP_ARCHIVE.mkdir(parents=True, exist_ok=True)

review_fields = [
    "candidate_group_number", "decision", "reason", "role",
    "title_match_type", "folder_path", "backup_canonical_path",
    "album_title", "album_uuid", "asset_count", "membership_checksum",
    "current_group_object_count", "backup_matching_album_count",
]

with DELETE_DUP_REVIEW.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=review_fields, delimiter="\t", lineterminator="\n")
    writer.writeheader()
    for row in review_rows:
        writer.writerow({key: delete_dup_tsv(row.get(key)) for key in review_fields})

manifest_fields = [
    "operation", "delete_album_uuid", "keep_album_uuid",
    "delete_folder_path", "keep_folder_path", "album_title",
    "normalized_title", "asset_count", "membership_checksum",
    "title_match_type", "backup_canonical_path",
    "candidate_group_number", "status",
]

with DELETE_DUP_MANIFEST.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=manifest_fields, delimiter="\t", lineterminator="\n")
    for row in manifest_rows:
        writer.writerow({key: delete_dup_tsv(row.get(key)) for key in manifest_fields})

archive_review = None
archive_manifest = None
if WRITE_DELETE_DUP_ARCHIVE:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    archive_review = DELETE_DUP_ARCHIVE / f"{timestamp}__delete_duplicate_candidates.tsv"
    archive_manifest = DELETE_DUP_ARCHIVE / f"{timestamp}__delete_duplicate_manifest.tsv"
    shutil.copy2(DELETE_DUP_REVIEW, archive_review)
    shutil.copy2(DELETE_DUP_MANIFEST, archive_manifest)

print("=" * 100)
print("Delete duplicate album manifest exported")
print("=" * 100)
print("Strict delete rows:", len(manifest_rows))
print("Candidate groups reviewed:", candidate_group_number)
print("Review TSV:", DELETE_DUP_REVIEW)
print("Executable manifest:", DELETE_DUP_MANIFEST)
if archive_review:
    print("Archive review:", archive_review)
    print("Archive manifest:", archive_manifest)
print()
print("Manifest fixed column order, no header:")
print("\t".join(manifest_fields))


## DELETE-02 — Post-Delete Audit

**Purpose:** Verify executed delete UUIDs are absent and keeper UUIDs, paths, titles, counts, and memberships remain intact.  
**Requires:** Freshly rebuilt Current Default inventory and the exact manifest that was executed.  
**Risk:** Read-only. **Run before overwriting that executed manifest with `DELETE-01`.**


In [ ]:
# ============================================================
# Post-delete audit — verify deleted albums absent
# and keeper albums intact
#
# READ ONLY:
# - reads the existing executed delete manifest
# - checks the freshly rebuilt inventory_current
# - does not modify Photos
# - does not overwrite the delete manifest
# ============================================================

from collections import defaultdict
from pathlib import Path
import csv
import hashlib


AUDIT_MANIFEST_PATH = (
    Path.home()
    / "Downloads"
    / "PhotosRepairMVP_Inbox"
    / "DeleteDuplicateAlbumsManifestLatest.tsv"
)

AUDIT_OUTPUT_PATH = (
    Path.home()
    / "Downloads"
    / "PhotosRepairMVP_Inbox"
    / "DeleteDuplicateAlbumsPostDeleteAuditLatest.tsv"
)


MANIFEST_FIELDS = [
    "operation",
    "delete_album_uuid",
    "keep_album_uuid",
    "delete_folder_path",
    "keep_folder_path",
    "album_title",
    "normalized_title",
    "asset_count",
    "membership_checksum",
    "title_match_type",
    "backup_canonical_path",
    "candidate_group_number",
    "status",
]


def audit_normalize_title(value):
    return " ".join(str(value or "").split())


def audit_normalize_path(value):
    text = (
        str(value or "")
        .strip()
        .replace("\\", "/")
        .replace(" / ", "/")
    )

    return " / ".join(
        part.strip()
        for part in text.split("/")
        if part.strip()
    )


def audit_leaf_paths(album):
    paths = {
        audit_normalize_path(
            folder.get("path")
            or folder.get("title")
        )
        for folder in (
            album.get("folders") or {}
        ).values()
        if folder.get("path")
        or folder.get("title")
    }

    paths.discard("")

    if not paths:
        return ("[ROOT_ALBUMS]",)

    return tuple(
        sorted(
            path
            for path in paths
            if not any(
                other != path
                and other.startswith(path + " / ")
                for other in paths
            )
        )
    )


def audit_checksum(values):
    payload = "\n".join(sorted(values))

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:16]


# ------------------------------------------------------------
# Read the executed manifest.
# It intentionally has no header.
# ------------------------------------------------------------

manifest_rows = []

with AUDIT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    reader = csv.DictReader(
        file,
        fieldnames=MANIFEST_FIELDS,
        delimiter="\t",
    )

    manifest_rows = [
        dict(row)
        for row in reader
        if any(
            str(value or "").strip()
            for value in row.values()
        )
    ]


# ------------------------------------------------------------
# Build live album and membership indexes from the freshly
# rebuilt inventory_current.
# ------------------------------------------------------------

albums_by_uuid = {}

for inventory_key, album in (
    inventory_current.get("albums") or {}
).items():
    album_uuid = str(
        album.get("uuid")
        or inventory_key
    )

    albums_by_uuid[album_uuid] = album


members_by_album_uuid = defaultdict(set)

for asset in (
    inventory_current.get("assets") or []
):
    asset_uuid = str(
        asset.get("uuid") or ""
    )

    if not asset_uuid:
        continue

    for album_uuid in (
        asset.get("albums") or {}
    ):
        members_by_album_uuid[
            str(album_uuid)
        ].add(asset_uuid)


# ------------------------------------------------------------
# Verify every executed manifest job.
# ------------------------------------------------------------

audit_rows = []

for job_number, manifest_row in enumerate(
    manifest_rows,
    start=1,
):
    delete_uuid = str(
        manifest_row["delete_album_uuid"]
    )

    keep_uuid = str(
        manifest_row["keep_album_uuid"]
    )

    expected_keep_path = (
        audit_normalize_path(
            manifest_row["keep_folder_path"]
        )
    )

    expected_normalized_title = (
        audit_normalize_title(
            manifest_row["normalized_title"]
        )
    )

    expected_asset_count = int(
        manifest_row["asset_count"]
    )

    expected_checksum = str(
        manifest_row["membership_checksum"]
    )

    delete_album = albums_by_uuid.get(
        delete_uuid
    )

    keep_album = albums_by_uuid.get(
        keep_uuid
    )

    delete_exists = (
        delete_album is not None
    )

    delete_member_refs = (
        members_by_album_uuid.get(
            delete_uuid,
            set(),
        )
    )

    delete_has_asset_references = bool(
        delete_member_refs
    )

    keep_exists = (
        keep_album is not None
    )

    if keep_exists:
        actual_keep_paths = (
            audit_leaf_paths(keep_album)
        )

        actual_keep_title = str(
            keep_album.get("title") or ""
        )

        actual_normalized_title = (
            audit_normalize_title(
                actual_keep_title
            )
        )

        actual_members = (
            members_by_album_uuid.get(
                keep_uuid,
                set(),
            )
        )

        actual_asset_count = len(
            actual_members
        )

        actual_checksum = audit_checksum(
            actual_members
        )
    else:
        actual_keep_paths = tuple()
        actual_keep_title = ""
        actual_normalized_title = ""
        actual_asset_count = -1
        actual_checksum = "-"

    keep_path_ok = (
        keep_exists
        and actual_keep_paths
        == (expected_keep_path,)
    )

    keep_title_ok = (
        keep_exists
        and actual_normalized_title
        == expected_normalized_title
    )

    keep_asset_count_ok = (
        keep_exists
        and actual_asset_count
        == expected_asset_count
    )

    keep_checksum_ok = (
        keep_exists
        and actual_checksum
        == expected_checksum
    )

    delete_absent_ok = (
        not delete_exists
    )

    delete_relationships_absent_ok = (
        not delete_has_asset_references
    )

    failure_reasons = []

    if not delete_absent_ok:
        failure_reasons.append(
            "DELETE_UUID_STILL_EXISTS"
        )

    if not delete_relationships_absent_ok:
        failure_reasons.append(
            "DELETE_UUID_STILL_REFERENCED_BY_ASSETS"
        )

    if not keep_exists:
        failure_reasons.append(
            "KEEPER_UUID_MISSING"
        )
    else:
        if not keep_path_ok:
            failure_reasons.append(
                "KEEPER_PATH_CHANGED"
            )

        if not keep_title_ok:
            failure_reasons.append(
                "KEEPER_TITLE_CHANGED"
            )

        if not keep_asset_count_ok:
            failure_reasons.append(
                "KEEPER_ASSET_COUNT_CHANGED"
            )

        if not keep_checksum_ok:
            failure_reasons.append(
                "KEEPER_MEMBERSHIP_CHANGED"
            )

    fully_passed = (
        not failure_reasons
    )

    audit_rows.append({
        "job_number": job_number,
        "candidate_group_number": (
            manifest_row[
                "candidate_group_number"
            ]
        ),
        "audit_status": (
            "PASS"
            if fully_passed
            else "FAIL"
        ),
        "failure_reasons": (
            " || ".join(failure_reasons)
            if failure_reasons
            else "-"
        ),
        "delete_album_uuid": delete_uuid,
        "delete_uuid_absent": (
            delete_absent_ok
        ),
        "delete_asset_relationships_absent": (
            delete_relationships_absent_ok
        ),
        "keep_album_uuid": keep_uuid,
        "keeper_exists": keep_exists,
        "expected_keep_folder_path": (
            expected_keep_path
        ),
        "actual_keep_folder_paths": (
            " || ".join(actual_keep_paths)
            if actual_keep_paths
            else "-"
        ),
        "keeper_path_ok": keep_path_ok,
        "keeper_title_ok": keep_title_ok,
        "expected_asset_count": (
            expected_asset_count
        ),
        "actual_asset_count": (
            actual_asset_count
        ),
        "keeper_asset_count_ok": (
            keep_asset_count_ok
        ),
        "expected_membership_checksum": (
            expected_checksum
        ),
        "actual_membership_checksum": (
            actual_checksum
        ),
        "keeper_membership_checksum_ok": (
            keep_checksum_ok
        ),
        "album_title": (
            manifest_row["album_title"]
        ),
    })


# ------------------------------------------------------------
# Export a readable audit TSV.
# ------------------------------------------------------------

AUDIT_FIELDS = [
    "job_number",
    "candidate_group_number",
    "audit_status",
    "failure_reasons",
    "delete_album_uuid",
    "delete_uuid_absent",
    "delete_asset_relationships_absent",
    "keep_album_uuid",
    "keeper_exists",
    "expected_keep_folder_path",
    "actual_keep_folder_paths",
    "keeper_path_ok",
    "keeper_title_ok",
    "expected_asset_count",
    "actual_asset_count",
    "keeper_asset_count_ok",
    "expected_membership_checksum",
    "actual_membership_checksum",
    "keeper_membership_checksum_ok",
    "album_title",
]

with AUDIT_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=AUDIT_FIELDS,
        delimiter="\t",
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(audit_rows)


# ------------------------------------------------------------
# Summary.
# ------------------------------------------------------------

passed_rows = [
    row
    for row in audit_rows
    if row["audit_status"] == "PASS"
]

failed_rows = [
    row
    for row in audit_rows
    if row["audit_status"] == "FAIL"
]


print("=" * 100)
print("Delete Duplicate Albums — Post-delete Audit")
print("=" * 100)
print("Manifest jobs:", len(audit_rows))
print("Fully passed:", len(passed_rows))
print("Failed:", len(failed_rows))
print()

print(
    "Delete UUID absent:",
    sum(
        bool(row["delete_uuid_absent"])
        for row in audit_rows
    ),
)

print(
    "Delete relationships absent:",
    sum(
        bool(
            row[
                "delete_asset_relationships_absent"
            ]
        )
        for row in audit_rows
    ),
)

print(
    "Keepers present:",
    sum(
        bool(row["keeper_exists"])
        for row in audit_rows
    ),
)

print(
    "Keeper paths correct:",
    sum(
        bool(row["keeper_path_ok"])
        for row in audit_rows
    ),
)

print(
    "Keeper titles correct:",
    sum(
        bool(row["keeper_title_ok"])
        for row in audit_rows
    ),
)

print(
    "Keeper asset counts correct:",
    sum(
        bool(
            row["keeper_asset_count_ok"]
        )
        for row in audit_rows
    ),
)

print(
    "Keeper membership checksums correct:",
    sum(
        bool(
            row[
                "keeper_membership_checksum_ok"
            ]
        )
        for row in audit_rows
    ),
)

print()
print("Audit TSV:", AUDIT_OUTPUT_PATH)

if failed_rows:
    print()
    print("FAILED JOBS:")

    for row in failed_rows:
        print(
            row["job_number"],
            row["failure_reasons"],
            row["delete_album_uuid"],
            row["keep_album_uuid"],
        )
else:
    print()
    print("SUCCESS: all manifest jobs passed post-delete audit.")

## REPORT-03 — Cross-Library Inventory Diff Summary

**Purpose:** Generate the cross-library inventory difference summary.  
**Requires:** `SETUP-02`.  
**Risk:** Read-only report generation.


In [ ]:
# ============================================================
# Cross-library inventory diff summary report
# ============================================================

diff_records, inventory_diff_summary_report = write_cross_library_inventory_diff_summary_report(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
    report_root=Path("reports/test2_inventory_diff_summary"),
    label="snapshot",
)

{
    "inventory_diff_summary_report": inventory_diff_summary_report,
}

## REPORT-04 — Assets Missing from Current Status

**Purpose:** Review/export assets classified as missing from Current.  
**Requires:** Cross-library comparison state.  
**Risk:** Read-only to Photos; optional file output.


In [ ]:
# ============================================================
# Status: ASSET_MISSING_FROM_CURRENT — DONE 20260615
# ============================================================

missing_current_review_result = print_missing_current_review(
    diff_records=diff_records,
    inventory_current=inventory_current,
    max_current_candidates=5,
)

# Optional: write text/TSV files only when you explicitly want files.
WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES = False

if WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES:
    missing_current_review_file_result = write_missing_current_review_report(
        diff_records=diff_records,
        inventory_current=inventory_current,
        output_dir=Path("reports/test2_missing_current_review"),
        report_name_prefix="asset_missing_from_current_review",
    )
    print_missing_current_review_report_summary(missing_current_review_file_result)


## REPAIR-01 — Find Folder/Album Relationship Repair Information

**Purpose:** Use Backup to identify Current albums whose folder relationships need repair.  
**Requires:** `SETUP-02`.  
**Risk:** Analysis only; does not modify Photos.


In [3]:
# =======================================================================
# Repare folder-album relations in current default using info in backup:
#   Find the info needed for reparments
# =======================================================================

from pathlib import Path
from datetime import datetime
import csv
import re
import shutil


# ------------------------------------------------------------
# User target
# ------------------------------------------------------------

TARGET_PARENT_FOLDER_PATH = "NSFW"

# ------------------------------------------------------------
# Path helpers
# ------------------------------------------------------------

def _normalize_photo_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]

    return " / ".join(parts)


def _join_photo_path(folder_path, album_title):
    folder = _normalize_photo_folder_path(folder_path)
    album = str(album_title).strip()

    if folder:
        return f"{folder} / {album}"

    return album


def _deepest_folder_path_for_album(album):
    folders = album.get("folders") or {}

    folder_paths = [
        _normalize_photo_folder_path(folder.get("path") or folder.get("title"))
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    ]

    folder_paths = [
        path
        for path in folder_paths
        if path
    ]

    if not folder_paths:
        return ""

    return sorted(
        folder_paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def _album_target_from_album(album):
    album_title = album.get("title")

    if not album_title:
        return None

    folder_path = _deepest_folder_path_for_album(album)
    album_title = str(album_title).strip()
    album_path = _join_photo_path(folder_path, album_title)

    return {
        "folder_path": folder_path,
        "album_title": album_title,
        "album_path": album_path,
    }


def _asset_album_targets(asset):
    targets = []

    for album in (asset.get("albums") or {}).values():
        target = _album_target_from_album(album)

        if target:
            targets.append(target)

    deduped = {}
    for target in targets:
        key = (
            target["folder_path"],
            target["album_title"],
            target["album_path"],
        )
        deduped[key] = target

    return list(deduped.values())


def _album_path_is_under_folder(album_path, parent_folder_path):
    parent = _normalize_photo_folder_path(parent_folder_path)

    return (
        album_path == parent
        or album_path.startswith(parent + " / ")
    )


def discover_backup_album_targets_under_folder(
    inventory_backup,
    target_parent_folder_path,
):
    target_parent = _normalize_photo_folder_path(target_parent_folder_path)

    rows_by_album_path = {}

    for asset in inventory_backup.get("assets") or []:
        for target in _asset_album_targets(asset):
            album_path = target["album_path"]

            if not _album_path_is_under_folder(album_path, target_parent):
                continue

            row = rows_by_album_path.setdefault(
                album_path,
                {
                    "folder_path": target["folder_path"],
                    "album_title": target["album_title"],
                    "album_path": target["album_path"],
                    "backup_assets": 0,
                    "backup_photos": 0,
                    "backup_videos": 0,
                },
            )

            row["backup_assets"] += 1

            if asset.get("is_movie"):
                row["backup_videos"] += 1
            else:
                row["backup_photos"] += 1

    rows = list(rows_by_album_path.values())

    rows.sort(
        key=lambda row: (
            -row["backup_assets"],
            row["album_path"],
        )
    )

    return rows


backup_album_targets = discover_backup_album_targets_under_folder(
    inventory_backup=inventory_backup,
    target_parent_folder_path=TARGET_PARENT_FOLDER_PATH,
)

print("=" * 120)
print("Backup album targets under folder:", _normalize_photo_folder_path(TARGET_PARENT_FOLDER_PATH))
print("=" * 120)
print("backup album target count:", len(backup_album_targets))
print()

print(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "idx",
        "backup_assets",
        "backup_photos",
        "backup_videos",
        "backup_album_path",
    )
)
print(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "---",
        "-------------",
        "-------------",
        "-------------",
        "-" * 80,
    )
)

for index, row in enumerate(backup_album_targets, start=1):
    print(
        "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
            index,
            row["backup_assets"],
            row["backup_photos"],
            row["backup_videos"],
            row["album_path"],
        )
    )

Backup album targets under folder: NSFW
backup album target count: 65

idx  backup_assets  backup_photos  backup_videos  backup_album_path
---  -------------  -------------  -------------  --------------------------------------------------------------------------------
  1           5029           5010             19  NSFW / Tumblr Girls - 1
  2           1594           1586              8  NSFW / IG Pretty Girls
  3           1074           1025             49  NSFW / Sexy Girls
  4            960              2            958  NSFW / Chaturbate
  5            897            858             39  NSFW / （隱藏內容）TG Girls -1
  6            773            754             19  NSFW / Pretty Girls
  7            387            387              0  NSFW / TG 色情CosPlay
  8            267            267              0  NSFW / Sexy nude photo albums - to be uploaded to Dropbox according to the comment and as the folder names
  9            196            196              0  NSFW / SM - NSFW
 10     

## REPAIR-02 — Preflight Existing Albums Elsewhere in Current

**Purpose:** Check whether proposed repair targets already exist at Root or another path before any create/move operation.  
**Requires:** `REPAIR-01`.  
**Risk:** Analysis only. This preflight is mandatory before exporting a repair manifest.


In [ ]:
# =======================================================================
# REPAIR-02 v2 — Preflight existing albums for MOVE_EXISTING_ALBUM_ONLY
#
# Purpose:
#   Find Current albums that can be moved into the Backup canonical folder
#   without creating albums and without changing asset membership.
#
# Safety:
#   - normalized title match
#   - exact cross-library membership match
#   - exactly one source album object
#   - exactly one source folder/root path
#   - no Photos modification
#   - writes review TSV + summary TXT
# =======================================================================

from collections import defaultdict, Counter
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import json
import re


PREFLIGHT_TARGET_PARENT_FOLDER_PATH = TARGET_PARENT_FOLDER_PATH
PREFLIGHT_REPORT_ROOT = Path(
    "reports/test2_scattered_current_album_preflight"
)
PREFLIGHT_MAX_PREVIEW_ROWS = 30
ROOT_ALBUMS_PATH = "[ROOT_ALBUMS]"


def _preflight_safe_filename_part(text):
    text = str(text).strip()
    text = text.replace(" / ", "__")
    text = text.replace("/", "__")
    text = re.sub(r"[^\w.-]+", "_", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text)
    return text.strip("_") or "untitled"


def _preflight_normalize_title(value):
    # Whitespace-only differences are not meaningful for matching.
    return " ".join(str(value or "").split())


def _preflight_normalize_path(value):
    return _normalize_photo_folder_path(value)


def _preflight_asset_unique_id_text(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        return None

    return json.dumps(
        tuple(unique_id),
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )


def _preflight_checksum(values):
    payload = "\n".join(sorted(str(value) for value in values))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def _preflight_album_leaf_folder_paths(album):
    folder_paths = {
        _preflight_normalize_path(
            folder.get("path") or folder.get("title")
        )
        for folder in (album.get("folders") or {}).values()
        if folder.get("path") or folder.get("title")
    }

    folder_paths.discard("")

    if not folder_paths:
        return (ROOT_ALBUMS_PATH,)

    leaf_paths = [
        path
        for path in folder_paths
        if not any(
            other != path
            and other.startswith(path + " / ")
            for other in folder_paths
        )
    ]

    return tuple(sorted(leaf_paths))


def _preflight_asset_album_paths(asset):
    album_paths = []

    for album in (asset.get("albums") or {}).values():
        target = _album_target_from_album(album)

        if target:
            album_paths.append(target["album_path"])

    return tuple(sorted(set(album_paths)))


def _preflight_asset_in_album_path(asset, album_path):
    return album_path in _preflight_asset_album_paths(asset)


def _preflight_build_album_members(inventory):
    cross_members_by_album_uuid = defaultdict(set)
    local_members_by_album_uuid = defaultdict(set)
    albums_with_missing_cross_identity = set()

    for asset in inventory.get("assets") or []:
        cross_identity = _preflight_asset_unique_id_text(asset)
        local_uuid = str(asset.get("uuid") or "")

        for album_uuid in (asset.get("albums") or {}):
            album_uuid = str(album_uuid)

            if cross_identity is None:
                albums_with_missing_cross_identity.add(album_uuid)
            else:
                cross_members_by_album_uuid[album_uuid].add(
                    cross_identity
                )

            if local_uuid:
                local_members_by_album_uuid[album_uuid].add(
                    local_uuid
                )

    return {
        "cross": cross_members_by_album_uuid,
        "local": local_members_by_album_uuid,
        "missing_cross_identity": albums_with_missing_cross_identity,
    }


def _preflight_backup_members_for_album_path(
    inventory_backup,
    album_path,
):
    members = set()
    missing_identity_count = 0

    for asset in inventory_backup.get("assets") or []:
        if not _preflight_asset_in_album_path(
            asset,
            album_path,
        ):
            continue

        identity = _preflight_asset_unique_id_text(asset)

        if identity is None:
            missing_identity_count += 1
        else:
            members.add(identity)

    return members, missing_identity_count


def check_target_backup_albums_scattered_in_current(
    *,
    inventory_backup,
    inventory_current,
    backup_album_targets,
    report_root,
    label,
):
    report_root = Path(report_root)
    report_root.mkdir(parents=True, exist_ok=True)

    run_time = datetime.now()
    report_dir = (
        report_root
        / f"{run_time:%Y%m%d-%H%M%S}__{label}"
    )
    report_dir.mkdir(parents=True, exist_ok=False)

    summary_path = (
        report_dir
        / "scattered_current_album_preflight_summary.txt"
    )
    review_tsv_path = (
        report_dir
        / "scattered_current_album_preflight.tsv"
    )

    current_member_indexes = _preflight_build_album_members(
        inventory_current
    )
    current_cross_members = current_member_indexes["cross"]
    current_local_members = current_member_indexes["local"]
    current_missing_identity_albums = (
        current_member_indexes["missing_cross_identity"]
    )

    current_albums_by_normalized_title = defaultdict(list)
    current_albums_by_cross_membership = defaultdict(list)

    for album in (inventory_current.get("albums") or {}).values():
        album_uuid = str(album.get("uuid") or "")
        raw_title = str(album.get("title") or "")
        normalized_title = _preflight_normalize_title(raw_title)
        cross_members = frozenset(
            current_cross_members.get(album_uuid, set())
        )

        if normalized_title:
            current_albums_by_normalized_title[
                normalized_title
            ].append(album)

        if cross_members:
            current_albums_by_cross_membership[
                cross_members
            ].append(album)

    rows = []

    for index, target in enumerate(
        backup_album_targets,
        start=1,
    ):
        backup_album_path = target["album_path"]
        backup_album_title = target["album_title"]
        normalized_title = _preflight_normalize_title(
            backup_album_title
        )
        target_folder_path = _preflight_normalize_path(
            target["folder_path"]
        )

        (
            backup_members,
            backup_missing_identity_count,
        ) = _preflight_backup_members_for_album_path(
            inventory_backup,
            backup_album_path,
        )

        same_title_candidates = (
            current_albums_by_normalized_title.get(
                normalized_title,
                [],
            )
        )

        exact_candidates = []

        for current_album in same_title_candidates:
            current_album_uuid = str(
                current_album.get("uuid") or ""
            )
            current_members = frozenset(
                current_cross_members.get(
                    current_album_uuid,
                    set(),
                )
            )

            if (
                current_members == frozenset(backup_members)
                and current_album_uuid
                not in current_missing_identity_albums
            ):
                exact_candidates.append(current_album)

        membership_only_candidates = (
            current_albums_by_cross_membership.get(
                frozenset(backup_members),
                [],
            )
            if backup_members
            else []
        )

        status = ""
        reason = ""
        source_album = None

        if backup_missing_identity_count:
            status = "BACKUP_ASSET_IDENTITY_MISSING"
            reason = (
                f"{backup_missing_identity_count} Backup assets "
                "lack cross-library identity"
            )

        elif not backup_members:
            status = "EMPTY_BACKUP_MEMBERSHIP"
            reason = "Backup album has no comparable members"

        elif len(exact_candidates) == 1:
            source_album = exact_candidates[0]
            source_paths = _preflight_album_leaf_folder_paths(
                source_album
            )

            if len(source_paths) != 1:
                status = "SOURCE_PATH_AMBIGUOUS"
                reason = (
                    f"source has {len(source_paths)} leaf paths"
                )

            elif source_paths[0] == target_folder_path:
                status = "ALREADY_AT_TARGET_PATH"
                reason = "unique exact match is already in target"

            else:
                status = "MOVE_READY"
                reason = (
                    "unique normalized-title and exact-membership "
                    "match exists outside target"
                )

        elif len(exact_candidates) > 1:
            status = "MULTIPLE_EXACT_MATCHES"
            reason = (
                f"{len(exact_candidates)} normalized-title "
                "and exact-membership matches"
            )

        elif same_title_candidates:
            status = "SAME_TITLE_MEMBERSHIP_MISMATCH"
            reason = (
                f"{len(same_title_candidates)} normalized-title "
                "candidate(s), none with exact membership"
            )

        elif len(membership_only_candidates) == 1:
            status = "UNIQUE_MEMBERSHIP_DIFFERENT_TITLE"
            reason = (
                "one exact-membership album exists, "
                "but normalized title differs"
            )

        elif len(membership_only_candidates) > 1:
            status = "MULTIPLE_MEMBERSHIP_ONLY_MATCHES"
            reason = (
                f"{len(membership_only_candidates)} exact-membership "
                "albums exist with different titles"
            )

        else:
            status = "NO_MATCHING_CURRENT_ALBUM"
            reason = (
                "no normalized-title plus exact-membership "
                "Current album exists"
            )

        source_album_uuid = ""
        source_album_title = ""
        source_folder_paths = ()
        source_asset_count = 0
        source_local_checksum = ""
        source_cross_checksum = ""

        if source_album is not None:
            source_album_uuid = str(
                source_album.get("uuid") or ""
            )
            source_album_title = str(
                source_album.get("title") or ""
            )
            source_folder_paths = (
                _preflight_album_leaf_folder_paths(
                    source_album
                )
            )
            source_local_members = current_local_members.get(
                source_album_uuid,
                set(),
            )
            source_cross_member_set = current_cross_members.get(
                source_album_uuid,
                set(),
            )
            source_asset_count = len(source_local_members)
            source_local_checksum = _preflight_checksum(
                source_local_members
            )
            source_cross_checksum = _preflight_checksum(
                source_cross_member_set
            )

        rows.append({
            "idx": index,
            "status": status,
            "reason": reason,
            "source_album_uuid": source_album_uuid,
            "source_folder_path": (
                source_folder_paths[0]
                if len(source_folder_paths) == 1
                else " || ".join(source_folder_paths)
            ),
            "source_path_count": len(source_folder_paths),
            "target_folder_path": target_folder_path,
            "source_album_title": source_album_title,
            "backup_album_title": backup_album_title,
            "normalized_title": normalized_title,
            "backup_album_path": backup_album_path,
            "backup_asset_count": len(backup_members),
            "source_asset_count": source_asset_count,
            "source_local_membership_checksum":
                source_local_checksum,
            "source_cross_membership_checksum":
                source_cross_checksum,
            "backup_cross_membership_checksum":
                _preflight_checksum(backup_members),
            "normalized_title_candidate_count":
                len(same_title_candidates),
            "exact_match_count": len(exact_candidates),
            "membership_only_candidate_count":
                len(membership_only_candidates),
        })

    status_counts = Counter(row["status"] for row in rows)
    move_source_path_counts = Counter(
        row["source_folder_path"]
        for row in rows
        if row["status"] == "MOVE_READY"
    )

    review_fields = [
        "idx",
        "status",
        "reason",
        "source_album_uuid",
        "source_folder_path",
        "source_path_count",
        "target_folder_path",
        "source_album_title",
        "backup_album_title",
        "normalized_title",
        "backup_album_path",
        "backup_asset_count",
        "source_asset_count",
        "source_local_membership_checksum",
        "source_cross_membership_checksum",
        "backup_cross_membership_checksum",
        "normalized_title_candidate_count",
        "exact_match_count",
        "membership_only_candidate_count",
    ]

    with review_tsv_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=review_fields,
            delimiter="\t",
            lineterminator="\n",
        )
        writer.writeheader()
        writer.writerows(rows)

    with summary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            "MOVE_EXISTING_ALBUM_ONLY preflight\n"
        )
        file.write("=" * 100 + "\n")
        file.write(
            f"run_timestamp\t"
            f"{run_time.isoformat(timespec='seconds')}\n"
        )
        file.write(
            f"target_parent_folder_path\t"
            f"{PREFLIGHT_TARGET_PARENT_FOLDER_PATH}\n"
        )
        file.write(
            f"backup_album_target_count\t"
            f"{len(backup_album_targets)}\n"
        )
        file.write("\nstatus_counts\n")
        for status, count in status_counts.most_common():
            file.write(f"{status}\t{count}\n")
        file.write("\nmove_source_path_counts\n")
        for path, count in move_source_path_counts.most_common():
            file.write(f"{path}\t{count}\n")
        file.write(
            f"\nreview_tsv\t{review_tsv_path}\n"
        )

    preview_rows = rows[:PREFLIGHT_MAX_PREVIEW_ROWS]

    result = {
        "target_parent_folder_path":
            PREFLIGHT_TARGET_PARENT_FOLDER_PATH,
        "backup_album_target_count":
            len(backup_album_targets),
        "status_counts": dict(status_counts),
        "move_source_path_counts":
            dict(move_source_path_counts),
        "summary_path": str(summary_path),
        "review_tsv_path": str(review_tsv_path),
        "rows": rows,
        "preview_rows": preview_rows,
    }

    print("=" * 100)
    print("MOVE_EXISTING_ALBUM_ONLY preflight")
    print("=" * 100)
    print("Target folder:", PREFLIGHT_TARGET_PARENT_FOLDER_PATH)
    print("Backup targets:", len(backup_album_targets))
    print("Status counts:", dict(status_counts))
    print(
        "MOVE_READY source paths:",
        dict(move_source_path_counts),
    )
    print("Summary:", summary_path)
    print("Review TSV:", review_tsv_path)

    return result


preflight_label = (
    "target_"
    + _preflight_safe_filename_part(
        PREFLIGHT_TARGET_PARENT_FOLDER_PATH
    )
)

scattered_current_album_preflight_result = (
    check_target_backup_albums_scattered_in_current(
        inventory_backup=inventory_backup,
        inventory_current=inventory_current,
        backup_album_targets=backup_album_targets,
        report_root=PREFLIGHT_REPORT_ROOT,
        label=preflight_label,
    )
)

scattered_current_album_preflight_result


## REPAIR-02B — Audit Backup Ground-Truth Album Structure

**Purpose:** Compare Backup and Current folder paths and album titles under the selected repair root, with whitespace-normalized matching.

**Requires:** `REPAIR-01`.

**Risk:** Analysis only. Writes a review TSV and summary TXT; does not modify Photos.


In [ ]:
# =============================================================================
# REPAIR-02B — Audit Backup Ground-Truth Album Structure
#
# Purpose:
#   Compare Backup ground-truth folder paths and album titles against Current.
#
# Matching:
#   - normalize leading/trailing whitespace
#   - collapse repeated whitespace
#   - preserve and report original raw titles
#
# Output:
#   - exact matches
#   - whitespace-only differences
#   - missing Current albums
#   - Current-only albums
#   - ambiguous normalized keys
#
# Safety:
#   - read-only
#   - no Photos modification
#   - writes review TSV + summary TXT
# =============================================================================

from collections import defaultdict, Counter
from pathlib import Path
from datetime import datetime
import csv
import re


STRUCTURE_AUDIT_TARGET_ROOT = TARGET_PARENT_FOLDER_PATH

STRUCTURE_AUDIT_REPORT_ROOT = Path(
    "reports/test2_backup_ground_truth_structure_audit"
)


def _structure_raw_text(value):
    return str(value or "")


def _structure_normalize_component(value):
    """
    Normalize whitespace for comparison only.

    Examples:
        "ABC  DEF"  -> "ABC DEF"
        " ABC DEF " -> "ABC DEF"

    This does not modify the actual Photos title.
    """
    return " ".join(_structure_raw_text(value).split())


def _structure_folder_components(path):
    text = _structure_raw_text(path)

    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    return [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]


def _structure_raw_folder_path(path):
    return " / ".join(
        _structure_folder_components(path)
    )


def _structure_normalized_folder_path(path):
    return " / ".join(
        _structure_normalize_component(part)
        for part in _structure_folder_components(path)
    )


def _structure_deepest_folder_path(album):
    folders = album.get("folders") or {}

    paths = []

    for folder in folders.values():
        raw_path = (
            folder.get("path")
            or folder.get("title")
            or ""
        )

        raw_path = _structure_raw_folder_path(raw_path)

        if raw_path:
            paths.append(raw_path)

    if not paths:
        return ""

    return sorted(
        set(paths),
        key=lambda value: (
            value.count(" / "),
            len(value),
            value,
        ),
    )[-1]


def _structure_is_under_root(
    folder_path,
    target_root,
):
    normalized_folder = (
        _structure_normalized_folder_path(folder_path)
    )

    normalized_root = (
        _structure_normalized_folder_path(target_root)
    )

    return (
        normalized_folder == normalized_root
        or normalized_folder.startswith(
            normalized_root + " / "
        )
    )


def _structure_collect_album_rows(
    inventory,
    target_root,
):
    rows = []

    for album in (
        inventory.get("albums") or {}
    ).values():
        raw_title = _structure_raw_text(
            album.get("title")
        )

        if not raw_title:
            continue

        raw_folder_path = (
            _structure_deepest_folder_path(album)
        )

        if not _structure_is_under_root(
            raw_folder_path,
            target_root,
        ):
            continue

        normalized_title = (
            _structure_normalize_component(
                raw_title
            )
        )

        normalized_folder_path = (
            _structure_normalized_folder_path(
                raw_folder_path
            )
        )

        rows.append({
            "album_uuid":
                str(album.get("uuid") or ""),
            "raw_folder_path":
                raw_folder_path,
            "normalized_folder_path":
                normalized_folder_path,
            "raw_title":
                raw_title,
            "normalized_title":
                normalized_title,
            "normalized_key": (
                normalized_folder_path,
                normalized_title,
            ),
            "exact_key": (
                raw_folder_path,
                raw_title,
            ),
        })

    rows.sort(
        key=lambda row: (
            row["normalized_folder_path"],
            row["normalized_title"],
            row["raw_folder_path"],
            row["raw_title"],
            row["album_uuid"],
        )
    )

    return rows


backup_structure_rows = (
    _structure_collect_album_rows(
        inventory_backup,
        STRUCTURE_AUDIT_TARGET_ROOT,
    )
)

current_structure_rows = (
    _structure_collect_album_rows(
        inventory_current,
        STRUCTURE_AUDIT_TARGET_ROOT,
    )
)


backup_by_normalized_key = defaultdict(list)
current_by_normalized_key = defaultdict(list)

for row in backup_structure_rows:
    backup_by_normalized_key[
        row["normalized_key"]
    ].append(row)

for row in current_structure_rows:
    current_by_normalized_key[
        row["normalized_key"]
    ].append(row)


all_normalized_keys = sorted(
    set(backup_by_normalized_key)
    | set(current_by_normalized_key)
)


comparison_rows = []

for normalized_key in all_normalized_keys:
    backup_matches = (
        backup_by_normalized_key.get(
            normalized_key,
            [],
        )
    )

    current_matches = (
        current_by_normalized_key.get(
            normalized_key,
            [],
        )
    )

    backup_exact_keys = {
        row["exact_key"]
        for row in backup_matches
    }

    current_exact_keys = {
        row["exact_key"]
        for row in current_matches
    }

    if backup_matches and not current_matches:
        status = "MISSING_IN_CURRENT"

    elif current_matches and not backup_matches:
        status = "CURRENT_ONLY"

    elif (
        len(backup_matches) != 1
        or len(current_matches) != 1
    ):
        status = "NORMALIZED_KEY_AMBIGUOUS"

    elif backup_exact_keys == current_exact_keys:
        status = "EXACT_MATCH"

    else:
        status = "WHITESPACE_ONLY_DIFFERENCE"

    comparison_rows.append({
        "status": status,
        "normalized_folder_path":
            normalized_key[0],
        "normalized_title":
            normalized_key[1],
        "backup_object_count":
            len(backup_matches),
        "current_object_count":
            len(current_matches),
        "backup_album_uuids":
            " || ".join(
                row["album_uuid"]
                for row in backup_matches
            ),
        "current_album_uuids":
            " || ".join(
                row["album_uuid"]
                for row in current_matches
            ),
        "backup_raw_folder_paths":
            " || ".join(
                sorted({
                    row["raw_folder_path"]
                    for row in backup_matches
                })
            ),
        "current_raw_folder_paths":
            " || ".join(
                sorted({
                    row["raw_folder_path"]
                    for row in current_matches
                })
            ),
        "backup_raw_titles":
            " || ".join(
                sorted({
                    row["raw_title"]
                    for row in backup_matches
                })
            ),
        "current_raw_titles":
            " || ".join(
                sorted({
                    row["raw_title"]
                    for row in current_matches
                })
            ),
    })


status_counts = Counter(
    row["status"]
    for row in comparison_rows
)


run_time = datetime.now()

report_dir = (
    STRUCTURE_AUDIT_REPORT_ROOT
    / (
        f"{run_time:%Y%m%d-%H%M%S}"
        f"__target_"
        f"{_structure_normalize_component(STRUCTURE_AUDIT_TARGET_ROOT)}"
    )
)

report_dir.mkdir(
    parents=True,
    exist_ok=False,
)

report_path = (
    report_dir
    / "backup_ground_truth_structure_audit.tsv"
)


report_fields = [
    "status",
    "normalized_folder_path",
    "normalized_title",
    "backup_object_count",
    "current_object_count",
    "backup_album_uuids",
    "current_album_uuids",
    "backup_raw_folder_paths",
    "current_raw_folder_paths",
    "backup_raw_titles",
    "current_raw_titles",
]


with report_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=report_fields,
        delimiter="\t",
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(comparison_rows)


print("=" * 100)
print(
    "BACKUP GROUND-TRUTH STRUCTURE AUDIT"
)
print("=" * 100)
print(
    "Target root:",
    _structure_normalized_folder_path(
        STRUCTURE_AUDIT_TARGET_ROOT
    ),
)
print(
    "Backup album objects:",
    len(backup_structure_rows),
)
print(
    "Current album objects:",
    len(current_structure_rows),
)
print()

print("Status counts")
print("-" * 100)

for status, count in sorted(
    status_counts.items()
):
    print(f"{status}: {count}")


print()
print(
    "Whitespace-only differences"
)
print("-" * 100)

whitespace_rows = [
    row
    for row in comparison_rows
    if row["status"]
        == "WHITESPACE_ONLY_DIFFERENCE"
]

if not whitespace_rows:
    print("None")
else:
    for index, row in enumerate(
        whitespace_rows,
        start=1,
    ):
        print(
            f"[{index}] "
            f"{row['normalized_folder_path']}"
            f" / "
            f"{row['normalized_title']}"
        )
        print(
            "  Backup folder:",
            repr(
                row[
                    "backup_raw_folder_paths"
                ]
            ),
        )
        print(
            "  Current folder:",
            repr(
                row[
                    "current_raw_folder_paths"
                ]
            ),
        )
        print(
            "  Backup title:",
            repr(
                row[
                    "backup_raw_titles"
                ]
            ),
        )
        print(
            "  Current title:",
            repr(
                row[
                    "current_raw_titles"
                ]
            ),
        )


print()
print(
    "Missing in Current after whitespace normalization"
)
print("-" * 100)

missing_rows = [
    row
    for row in comparison_rows
    if row["status"] == "MISSING_IN_CURRENT"
]

if not missing_rows:
    print("None")
else:
    for index, row in enumerate(
        missing_rows,
        start=1,
    ):
        print(
            f"[{index}] "
            f"{row['normalized_folder_path']}"
            f" / "
            f"{row['normalized_title']}"
        )
        print(
            "  Backup UUID:",
            row["backup_album_uuids"],
        )
        print(
            "  Backup raw title:",
            repr(row["backup_raw_titles"]),
        )


print()
print(
    "Ambiguous normalized keys"
)
print("-" * 100)

ambiguous_rows = [
    row
    for row in comparison_rows
    if row["status"]
        == "NORMALIZED_KEY_AMBIGUOUS"
]

if not ambiguous_rows:
    print("None")
else:
    for index, row in enumerate(
        ambiguous_rows,
        start=1,
    ):
        print(
            f"[{index}] "
            f"{row['normalized_folder_path']}"
            f" / "
            f"{row['normalized_title']}"
        )
        print(
            "  Backup object count:",
            row["backup_object_count"],
        )
        print(
            "  Current object count:",
            row["current_object_count"],
        )
        print(
            "  Backup UUIDs:",
            row["backup_album_uuids"],
        )
        print(
            "  Current UUIDs:",
            row["current_album_uuids"],
        )


print()
print("Full TSV:", report_path)


backup_ground_truth_structure_audit_result = {
    "backup_rows":
        backup_structure_rows,
    "current_rows":
        current_structure_rows,
    "comparison_rows":
        comparison_rows,
    "status_counts":
        status_counts,
    "missing_rows":
        missing_rows,
    "whitespace_rows":
        whitespace_rows,
    "ambiguous_rows":
        ambiguous_rows,
    "report_path":
        report_path,
}

## REPAIR-02C — Audit Root-to-Canonical-Path Album Collisions

**Purpose:** For the repair root selected in `REPAIR-01`, find only Current **root albums** whose whitespace-normalized title also exists at a Backup canonical target path under that repair root. Compare their Current memberships so root-to-canonical collisions can be classified without mixing in unrelated folders such as `HIDE`.

**Requires:** `SETUP-02` and `REPAIR-01`. This audit is independent of `REPAIR-02` and `REPAIR-02B`.

**Output:** A project-local summary and TSVs under `reports/test2_root_to_canonical_album_collision_audit/`.

**Risk:** Analysis only. Does not modify Photos and does not write an executable manifest.

**Important:** This section diagnoses a different scenario from `REPAIR-02`:
- `REPAIR-02` answers whether an existing album can be moved to a missing canonical path.
- `REPAIR-02C` answers whether a canonical target already exists **and** a same-title source album remains at `[ROOT_ALBUMS]`.


In [ ]:
# =============================================================================
# REPAIR-02C — Audit Root-to-Canonical-Path Album Collisions
#
# Scope:
#   - Backup targets created by REPAIR-01 under TARGET_PARENT_FOLDER_PATH
#   - Current album at the Backup canonical folder path
#   - Current root album with the same whitespace-normalized title
#
# Deliberately excluded:
#   - same-title albums under unrelated folders such as HIDE
#   - executable manifest creation
#   - all Photos modifications
#
# Output:
#   reports/test2_root_to_canonical_album_collision_audit/
#     <timestamp>__target_<scope>/
#       root_to_canonical_target_summary.tsv
#       root_to_canonical_collision_pairs.tsv
#       root_to_canonical_collision_summary.txt
# =============================================================================

from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import csv
import hashlib
import re


ROOT_CANONICAL_AUDIT_SCOPE = _normalize_photo_folder_path(
    TARGET_PARENT_FOLDER_PATH
)

ROOT_CANONICAL_AUDIT_REPORT_ROOT = Path(
    "reports/test2_root_to_canonical_album_collision_audit"
)

ROOT_ALBUMS_PATH = "[ROOT_ALBUMS]"


def _r02c_safe_filename_part(value):
    value = str(value).strip().replace(" / ", "__").replace("/", "__")
    value = re.sub(r"[^\w.-]+", "_", value, flags=re.UNICODE)
    return re.sub(r"_+", "_", value).strip("_") or "untitled"


def _r02c_normalize_title(value):
    # Only whitespace differences are ignored for title matching.
    return " ".join(str(value or "").split())


def _r02c_leaf_folder_paths(album):
    folder_paths = {
        _normalize_photo_folder_path(
            folder.get("path") or folder.get("title")
        )
        for folder in (album.get("folders") or {}).values()
        if folder.get("path") or folder.get("title")
    }
    folder_paths.discard("")

    if not folder_paths:
        return (ROOT_ALBUMS_PATH,)

    leaf_paths = [
        path
        for path in folder_paths
        if not any(
            other != path
            and other.startswith(path + " / ")
            for other in folder_paths
        )
    ]

    return tuple(sorted(leaf_paths))


def _r02c_checksum(values):
    payload = "\n".join(sorted(str(value) for value in values))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def _r02c_display_album_path(folder_path, title):
    if folder_path == ROOT_ALBUMS_PATH:
        return f"{ROOT_ALBUMS_PATH} / {title}"

    return f"{folder_path} / {title}"


def _r02c_membership_relation(root_members, canonical_members):
    if root_members == canonical_members:
        return "EXACT_SAME_MEMBERSHIP"

    if root_members <= canonical_members:
        return "ROOT_SUBSET_OF_CANONICAL"

    if canonical_members <= root_members:
        return "ROOT_SUPERSET_OF_CANONICAL"

    if root_members & canonical_members:
        return "OVERLAP_WITH_BOTH_SIDES_UNIQUE"

    return "DISJOINT_MEMBERSHIP"


def _r02c_recommended_action(relation):
    actions = {
        "EXACT_SAME_MEMBERSHIP": (
            "STRICT_DUPLICATE_DELETE_CANDIDATE"
        ),
        "ROOT_SUBSET_OF_CANONICAL": (
            "MANUAL_REVIEW_ROOT_MEMBERS_ALREADY_PRESERVED"
        ),
        "ROOT_SUPERSET_OF_CANONICAL": (
            "MANUAL_ADDITIVE_MERGE_ROOT_TO_CANONICAL_REQUIRED"
        ),
        "OVERLAP_WITH_BOTH_SIDES_UNIQUE": (
            "MANUAL_ADDITIVE_MERGE_ROOT_TO_CANONICAL_REQUIRED"
        ),
        "DISJOINT_MEMBERSHIP": (
            "MANUAL_REVIEW_DO_NOT_MERGE_AUTOMATICALLY"
        ),
    }

    return actions[relation]


def _r02c_clean_tsv(value):
    if value is None:
        return "-"

    return (
        str(value)
        .replace("\t", " ")
        .replace("\r", " ")
        .replace("\n", " ")
    )


# -------------------------------------------------------------------------
# Build exact Current-local membership sets for all Current albums.
# -------------------------------------------------------------------------

current_local_members_by_album_uuid = defaultdict(set)

for asset in inventory_current.get("assets") or []:
    asset_uuid = str(asset.get("uuid") or "")

    if not asset_uuid:
        continue

    for album_uuid in (asset.get("albums") or {}):
        current_local_members_by_album_uuid[
            str(album_uuid)
        ].add(asset_uuid)


# -------------------------------------------------------------------------
# Index Current album objects by normalized title and concrete path.
# -------------------------------------------------------------------------

current_album_rows_by_title = defaultdict(list)

for inventory_key, album in (
    inventory_current.get("albums") or {}
).items():
    album_uuid = str(album.get("uuid") or inventory_key)
    raw_title = str(album.get("title") or "")
    normalized_title = _r02c_normalize_title(raw_title)

    if not normalized_title:
        continue

    leaf_paths = _r02c_leaf_folder_paths(album)
    local_members = set(
        current_local_members_by_album_uuid.get(
            album_uuid,
            set(),
        )
    )

    current_album_rows_by_title[normalized_title].append({
        "album_uuid": album_uuid,
        "raw_title": raw_title,
        "normalized_title": normalized_title,
        "leaf_paths": leaf_paths,
        "local_members": local_members,
        "asset_count": len(local_members),
        "membership_checksum": _r02c_checksum(local_members),
    })


# -------------------------------------------------------------------------
# Compare each Backup canonical target only with Current root candidates.
# -------------------------------------------------------------------------

run_time = datetime.now()
report_dir = (
    ROOT_CANONICAL_AUDIT_REPORT_ROOT
    / (
        f"{run_time:%Y%m%d-%H%M%S}__target_"
        f"{_r02c_safe_filename_part(ROOT_CANONICAL_AUDIT_SCOPE)}"
    )
)
report_dir.mkdir(parents=True, exist_ok=False)

summary_tsv_path = (
    report_dir / "root_to_canonical_target_summary.tsv"
)
pairs_tsv_path = (
    report_dir / "root_to_canonical_collision_pairs.tsv"
)
summary_txt_path = (
    report_dir / "root_to_canonical_collision_summary.txt"
)

target_summary_rows = []
collision_pair_rows = []

for target_index, target in enumerate(
    backup_album_targets,
    start=1,
):
    canonical_folder_path = _normalize_photo_folder_path(
        target["folder_path"]
    )
    backup_album_title = str(target["album_title"] or "")
    normalized_title = _r02c_normalize_title(
        backup_album_title
    )
    backup_album_path = str(target["album_path"] or "")

    title_candidates = current_album_rows_by_title.get(
        normalized_title,
        [],
    )

    canonical_candidates = [
        row
        for row in title_candidates
        if row["leaf_paths"] == (canonical_folder_path,)
    ]

    root_candidates = [
        row
        for row in title_candidates
        if row["leaf_paths"] == (ROOT_ALBUMS_PATH,)
    ]

    canonical_count = len(canonical_candidates)
    root_count = len(root_candidates)

    if root_count == 0:
        target_status = "NO_ROOT_SAME_TITLE_CANDIDATE"
    elif canonical_count == 0:
        target_status = "ROOT_SAME_TITLE_BUT_CANONICAL_TARGET_MISSING"
    elif canonical_count == 1:
        target_status = "ROOT_TO_CANONICAL_COLLISION"
    else:
        target_status = (
            "ROOT_SAME_TITLE_AND_MULTIPLE_CANONICAL_TARGETS"
        )

    target_summary_rows.append({
        "target_index": target_index,
        "target_status": target_status,
        "backup_album_path": backup_album_path,
        "canonical_folder_path": canonical_folder_path,
        "backup_album_title": backup_album_title,
        "normalized_title": normalized_title,
        "backup_asset_count": int(
            target.get("backup_assets") or 0
        ),
        "current_canonical_album_count": canonical_count,
        "current_root_same_title_album_count": root_count,
    })

    if canonical_count != 1 or root_count == 0:
        continue

    canonical = canonical_candidates[0]
    canonical_members = canonical["local_members"]

    for root in root_candidates:
        root_members = root["local_members"]
        common_members = root_members & canonical_members
        root_only_members = root_members - canonical_members
        canonical_only_members = canonical_members - root_members

        relation = _r02c_membership_relation(
            root_members,
            canonical_members,
        )

        collision_pair_rows.append({
            "target_index": target_index,
            "backup_album_path": backup_album_path,
            "canonical_folder_path": canonical_folder_path,
            "normalized_title": normalized_title,
            "backup_album_title": backup_album_title,
            "canonical_album_path": _r02c_display_album_path(
                canonical_folder_path,
                canonical["raw_title"],
            ),
            "canonical_album_uuid": canonical["album_uuid"],
            "canonical_raw_title": canonical["raw_title"],
            "canonical_asset_count": canonical["asset_count"],
            "canonical_membership_checksum": (
                canonical["membership_checksum"]
            ),
            "root_album_path": _r02c_display_album_path(
                ROOT_ALBUMS_PATH,
                root["raw_title"],
            ),
            "root_album_uuid": root["album_uuid"],
            "root_raw_title": root["raw_title"],
            "root_asset_count": root["asset_count"],
            "root_membership_checksum": (
                root["membership_checksum"]
            ),
            "common_asset_count": len(common_members),
            "root_only_asset_count": len(root_only_members),
            "canonical_only_asset_count": len(
                canonical_only_members
            ),
            "membership_relation": relation,
            "recommended_action": _r02c_recommended_action(
                relation
            ),
        })


target_summary_rows.sort(
    key=lambda row: (
        row["backup_album_path"],
        row["target_index"],
    )
)

collision_pair_rows.sort(
    key=lambda row: (
        row["membership_relation"],
        row["backup_album_path"],
        row["root_album_uuid"],
    )
)

summary_fields = [
    "target_index",
    "target_status",
    "backup_album_path",
    "canonical_folder_path",
    "backup_album_title",
    "normalized_title",
    "backup_asset_count",
    "current_canonical_album_count",
    "current_root_same_title_album_count",
]

pair_fields = [
    "target_index",
    "backup_album_path",
    "canonical_folder_path",
    "normalized_title",
    "backup_album_title",
    "canonical_album_path",
    "canonical_album_uuid",
    "canonical_raw_title",
    "canonical_asset_count",
    "canonical_membership_checksum",
    "root_album_path",
    "root_album_uuid",
    "root_raw_title",
    "root_asset_count",
    "root_membership_checksum",
    "common_asset_count",
    "root_only_asset_count",
    "canonical_only_asset_count",
    "membership_relation",
    "recommended_action",
]

with summary_tsv_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=summary_fields,
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    for row in target_summary_rows:
        writer.writerow({
            key: _r02c_clean_tsv(row.get(key))
            for key in summary_fields
        })

with pairs_tsv_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=pair_fields,
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    for row in collision_pair_rows:
        writer.writerow({
            key: _r02c_clean_tsv(row.get(key))
            for key in pair_fields
        })

target_status_counts = Counter(
    row["target_status"]
    for row in target_summary_rows
)
relation_counts = Counter(
    row["membership_relation"]
    for row in collision_pair_rows
)

with summary_txt_path.open(
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "ROOT-TO-CANONICAL-PATH ALBUM COLLISION AUDIT\n"
    )
    file.write("=" * 100 + "\n")
    file.write(
        f"run_timestamp\t"
        f"{run_time.isoformat(timespec='seconds')}\n"
    )
    file.write(
        f"target_parent_folder_path\t"
        f"{ROOT_CANONICAL_AUDIT_SCOPE}\n"
    )
    file.write(
        f"backup_target_count\t"
        f"{len(target_summary_rows)}\n"
    )
    file.write(
        f"root_to_canonical_pair_count\t"
        f"{len(collision_pair_rows)}\n"
    )
    file.write("\ntarget_status_counts\n")
    for status, count in target_status_counts.most_common():
        file.write(f"{status}\t{count}\n")
    file.write("\nmembership_relation_counts\n")
    for relation, count in relation_counts.most_common():
        file.write(f"{relation}\t{count}\n")
    file.write(f"\ntarget_summary_tsv\t{summary_tsv_path}\n")
    file.write(f"collision_pairs_tsv\t{pairs_tsv_path}\n")

root_to_canonical_collision_audit_result = {
    "target_parent_folder_path": ROOT_CANONICAL_AUDIT_SCOPE,
    "backup_target_count": len(target_summary_rows),
    "root_to_canonical_pair_count": len(collision_pair_rows),
    "target_status_counts": dict(target_status_counts),
    "membership_relation_counts": dict(relation_counts),
    "target_summary_tsv": str(summary_tsv_path),
    "collision_pairs_tsv": str(pairs_tsv_path),
    "summary_txt": str(summary_txt_path),
    "target_summary_rows": target_summary_rows,
    "collision_pair_rows": collision_pair_rows,
}

print("=" * 100)
print("REPAIR-02C — Root-to-canonical-path album collision audit")
print("=" * 100)
print("Target folder:", ROOT_CANONICAL_AUDIT_SCOPE)
print("Backup targets:", len(target_summary_rows))
print("Root-to-canonical collision pairs:", len(collision_pair_rows))
print("Target status counts:", dict(target_status_counts))
print("Membership relation counts:", dict(relation_counts))
print("Target summary TSV:", summary_tsv_path)
print("Collision pairs TSV:", pairs_tsv_path)
print("Summary:", summary_txt_path)

if collision_pair_rows:
    print()
    print("Collision preview:")
    for row in collision_pair_rows[:20]:
        print(
            f"- {row['membership_relation']} | "
            f"root={row['root_asset_count']} | "
            f"canonical={row['canonical_asset_count']} | "
            f"root-only={row['root_only_asset_count']} | "
            f"canonical-only={row['canonical_only_asset_count']} | "
            f"{row['backup_album_path']}"
        )


## REPAIR-03 — Export PhotosRepairMVP Move-Existing-Albums Manifest

**Purpose:** Export the TSV consumed by the MoveExistingAlbumsOnly PhotosRepairMVP executor.

**Requires:** `REPAIR-01` and the narrow `REPAIR-02` move-existing preflight, with results reviewed.

**Scope:** Exports only strict `MOVE_READY` rows: a unique existing Current album can be moved unchanged because the Backup canonical target path does not already contain that album.

**Risk:** **High downstream risk.** This cell only writes TSV, but the paired macOS app changes the Default Current Photos Library.

**Do not use for:** Root-to-canonical collisions where the canonical target already exists. Use `REPAIR-02C` to diagnose those cases.


In [ ]:
# =======================================================================
# REPAIR-03 v2 — Export MOVE_EXISTING_ALBUM_ONLY manifest
#
# This manifest contains one row per existing Current album object.
# It never asks the macOS app to create an album or add/remove assets.
#
# Requires:
#   REPAIR-02 v2 completed and reviewed.
#
# First test:
#   MAX_MOVES_TO_EXPORT = 1
#
# After the verified one-album execution:
#   export every strict MOVE_READY row; the Swift app runs them in verified batches.
# =======================================================================

from pathlib import Path
from datetime import datetime
import csv
import shutil


MOVE_INBOX_DIR = (
    Path.home()
    / "Downloads"
    / "PhotosRepairMVP_Inbox"
)

MOVE_MANIFEST_LATEST = (
    MOVE_INBOX_DIR
    / "MoveExistingAlbumsManifestLatest.tsv"
)

MOVE_REVIEW_LATEST = (
    MOVE_INBOX_DIR
    / "MoveExistingAlbumsReviewLatest.tsv"
)

MOVE_ARCHIVE_DIR = MOVE_INBOX_DIR / "archive"
WRITE_MOVE_ARCHIVE = True

# Safety scope for the current 股票 repair.
REQUIRE_SOURCE_FOLDER_PATH = "[ROOT_ALBUMS]"
REQUIRE_TARGET_FOLDER_PATH = _normalize_photo_folder_path(
    TARGET_PARENT_FOLDER_PATH
)

# Export from the MOVE_READY list, not from all Backup targets.
MOVE_START_INDEX = 0

# Export every strict MOVE_READY row.
# The already-moved first row may still be present in stale preflight data;
# the Swift app safely classifies it as ALREADY MOVED.
MAX_MOVES_TO_EXPORT = None


def _move_tsv_clean(value):
    if value is None:
        return "-"

    return (
        str(value)
        .replace("\t", " ")
        .replace("\r", " ")
        .replace("\n", " ")
    )


preflight_rows = list(
    scattered_current_album_preflight_result.get(
        "rows",
        [],
    )
)

move_ready_rows = [
    row
    for row in preflight_rows
    if (
        row.get("status") == "MOVE_READY"
        and row.get("source_folder_path")
            == REQUIRE_SOURCE_FOLDER_PATH
        and _normalize_photo_folder_path(
            row.get("target_folder_path")
        )
            == REQUIRE_TARGET_FOLDER_PATH
        and int(row.get("source_path_count") or 0) == 1
        and row.get("source_album_uuid")
        and int(row.get("source_asset_count") or 0)
            == int(row.get("backup_asset_count") or 0)
        and row.get(
            "source_cross_membership_checksum"
        )
            == row.get(
                "backup_cross_membership_checksum"
            )
    )
]

move_ready_rows.sort(
    key=lambda row: (
        row.get("source_folder_path") or "",
        row.get("normalized_title") or "",
        row.get("source_album_uuid") or "",
    )
)

source_uuids = [
    row["source_album_uuid"]
    for row in move_ready_rows
]

if len(source_uuids) != len(set(source_uuids)):
    raise RuntimeError(
        "Duplicate source_album_uuid in MOVE_READY rows."
    )

if MAX_MOVES_TO_EXPORT is None:
    selected_move_rows = move_ready_rows[
        MOVE_START_INDEX:
    ]
else:
    selected_move_rows = move_ready_rows[
        MOVE_START_INDEX:
        MOVE_START_INDEX + MAX_MOVES_TO_EXPORT
    ]

manifest_rows = []

for sequence, row in enumerate(
    selected_move_rows,
    start=1,
):
    manifest_rows.append({
        "operation": "move_existing_album",
        "sequence": sequence,
        "source_album_uuid":
            row["source_album_uuid"],
        "source_folder_path":
            row["source_folder_path"],
        "target_folder_path":
            _normalize_photo_folder_path(
                row["target_folder_path"]
            ),
        "source_album_title":
            row["source_album_title"],
        "backup_album_title":
            row["backup_album_title"],
        "normalized_title":
            row["normalized_title"],
        "asset_count":
            row["source_asset_count"],
        "local_membership_checksum":
            row[
                "source_local_membership_checksum"
            ],
        "cross_membership_checksum":
            row[
                "source_cross_membership_checksum"
            ],
        "source_path_count":
            row["source_path_count"],
        "preflight_status":
            row["status"],
    })

manifest_fields = [
    "operation",
    "sequence",
    "source_album_uuid",
    "source_folder_path",
    "target_folder_path",
    "source_album_title",
    "backup_album_title",
    "normalized_title",
    "asset_count",
    "local_membership_checksum",
    "cross_membership_checksum",
    "source_path_count",
    "preflight_status",
]

review_fields = [
    "idx",
    "status",
    "reason",
    "source_album_uuid",
    "source_folder_path",
    "target_folder_path",
    "source_album_title",
    "backup_album_title",
    "normalized_title",
    "backup_asset_count",
    "source_asset_count",
    "source_local_membership_checksum",
    "source_cross_membership_checksum",
    "backup_cross_membership_checksum",
]

MOVE_INBOX_DIR.mkdir(parents=True, exist_ok=True)
MOVE_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

with MOVE_REVIEW_LATEST.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=review_fields,
        delimiter="\t",
        lineterminator="\n",
        extrasaction="ignore",
    )
    writer.writeheader()
    for row in preflight_rows:
        writer.writerow({
            key: _move_tsv_clean(row.get(key))
            for key in review_fields
        })

with MOVE_MANIFEST_LATEST.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=manifest_fields,
        delimiter="\t",
        lineterminator="\n",
        extrasaction="ignore",
    )
    writer.writeheader()
    for row in manifest_rows:
        writer.writerow({
            key: _move_tsv_clean(row.get(key))
            for key in manifest_fields
        })

archive_manifest = None
archive_review = None

if WRITE_MOVE_ARCHIVE:
    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    archive_manifest = (
        MOVE_ARCHIVE_DIR
        / (
            f"{timestamp}__"
            f"move_existing_albums_"
            f"{len(manifest_rows)}.tsv"
        )
    )
    archive_review = (
        MOVE_ARCHIVE_DIR
        / (
            f"{timestamp}__"
            "move_existing_albums_review.tsv"
        )
    )
    shutil.copy2(
        MOVE_MANIFEST_LATEST,
        archive_manifest,
    )
    shutil.copy2(
        MOVE_REVIEW_LATEST,
        archive_review,
    )

print("=" * 100)
print("MOVE_EXISTING_ALBUM_ONLY manifest exported")
print("=" * 100)
print("All preflight rows:", len(preflight_rows))
print("All strict MOVE_READY rows:", len(move_ready_rows))
print("Rows exported now:", len(manifest_rows))
print("Source required:", REQUIRE_SOURCE_FOLDER_PATH)
print("Target required:", REQUIRE_TARGET_FOLDER_PATH)
print("Manifest:", MOVE_MANIFEST_LATEST)
print("Review:", MOVE_REVIEW_LATEST)

if archive_manifest:
    print("Archive manifest:", archive_manifest)
    print("Archive review:", archive_review)

print()
print("Manifest columns:")
print("\t".join(manifest_fields))

if not manifest_rows:
    print()
    print("STOP: zero executable move rows were exported.")

move_existing_album_manifest_export_result = {
    "all_preflight_rows": preflight_rows,
    "all_move_ready_rows": move_ready_rows,
    "manifest_rows": manifest_rows,
    "manifest_path": MOVE_MANIFEST_LATEST,
    "review_path": MOVE_REVIEW_LATEST,
    "archive_manifest_path": archive_manifest,
    "archive_review_path": archive_review,
}


## REPAIR-04 — Export Create-Missing-Albums Manifest

**Purpose:** Export a strict TSV for Backup albums missing from Current under the selected repair root, excluding `#給資料夾置頂用`.

**Requires:** `SETUP-02`, `REPAIR-01`, and the completed existing-album move repair.

**Risk:** **High downstream risk.** This cell only writes TSV files; the paired macOS app creates albums and adds existing Current assets.


In [ ]:
# =======================================================================
# REPAIR-04 — Export CREATE_MISSING_ALBUM_WITH_EXISTING_ASSETS manifest
#
# Ground truth:
#   Backup folder/album structure under TARGET_PARENT_FOLDER_PATH.
#
# Selection:
#   - compare normalized folder path + normalized album title
#   - ignore whitespace-only differences
#   - exclude "#給資料夾置頂用"
#   - export only albums absent from Current at the canonical target path
#
# Asset safety:
#   - every Backup album member must map uniquely to one Current asset by
#     photo_library_asset_unique_id
#   - any missing/ambiguous asset aborts the entire export
#
# Output:
#   ~/Downloads/PhotosRepairMVP_Inbox/
#       CreateMissingAlbumsManifestLatest.tsv
#       CreateMissingAlbumsReviewLatest.tsv
#       archive/<timestamp>__create_missing_albums_<count>.tsv
#
# This cell does not modify Photos.
# =======================================================================

from collections import defaultdict, Counter
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import shutil


# -----------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------

CREATE_TARGET_ROOT = _normalize_photo_folder_path(
    TARGET_PARENT_FOLDER_PATH
)

CREATE_EXCLUDED_NORMALIZED_TITLES = {
    " ".join("#給資料夾置頂用".split()),
}

CREATE_INBOX_DIR = (
    Path.home()
    / "Downloads"
    / "PhotosRepairMVP_Inbox"
)

CREATE_MANIFEST_LATEST = (
    CREATE_INBOX_DIR
    / "CreateMissingAlbumsManifestLatest.tsv"
)

CREATE_REVIEW_LATEST = (
    CREATE_INBOX_DIR
    / "CreateMissingAlbumsReviewLatest.tsv"
)

CREATE_ARCHIVE_DIR = CREATE_INBOX_DIR / "archive"
WRITE_CREATE_ARCHIVE = True


# -----------------------------------------------------------------------
# Normalization and inventory helpers
# -----------------------------------------------------------------------

def _create_normalize_title(value):
    return " ".join(str(value or "").split())


def _create_normalize_path(value):
    return _normalize_photo_folder_path(value)


def _create_clean_tsv(value):
    if value is None:
        return "-"

    return (
        str(value)
        .replace("\t", " ")
        .replace("\r", " ")
        .replace("\n", " ")
    )


def _create_album_leaf_path(album):
    return _create_normalize_path(
        _deepest_folder_path_for_album(album)
    )


def _create_album_key(album):
    raw_title = str(album.get("title") or "").strip()

    if not raw_title:
        return None

    folder_path = _create_album_leaf_path(album)

    return (
        _create_normalize_path(folder_path),
        _create_normalize_title(raw_title),
    )


def _create_is_under_target(folder_path, target_root):
    folder_path = _create_normalize_path(folder_path)
    target_root = _create_normalize_path(target_root)

    return (
        folder_path == target_root
        or folder_path.startswith(target_root + " / ")
    )


def _create_members_by_album_uuid(inventory):
    members = defaultdict(list)

    for asset in inventory.get("assets") or []:
        for album_uuid in (asset.get("albums") or {}):
            members[str(album_uuid)].append(asset)

    return members


def _create_current_asset_index(inventory):
    index = defaultdict(list)

    for asset in inventory.get("assets") or []:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            continue

        index[tuple(unique_id)].append(asset)

    return index


def _create_membership_checksum(current_uuids):
    payload = "\n".join(sorted(str(value) for value in current_uuids))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


# -----------------------------------------------------------------------
# Build canonical Backup album catalog and Current target-path catalog
# -----------------------------------------------------------------------

backup_members_by_album_uuid = _create_members_by_album_uuid(
    inventory_backup
)

current_asset_index = _create_current_asset_index(
    inventory_current
)

backup_album_rows = []
backup_key_counts = Counter()

for album in (inventory_backup.get("albums") or {}).values():
    key = _create_album_key(album)

    if key is None:
        continue

    folder_path, normalized_title = key

    if not _create_is_under_target(
        folder_path,
        CREATE_TARGET_ROOT,
    ):
        continue

    backup_key_counts[key] += 1

    backup_album_rows.append({
        "backup_album_uuid": str(album.get("uuid") or ""),
        "target_folder_path": folder_path,
        "album_title": str(album.get("title") or "").strip(),
        "normalized_title": normalized_title,
        "key": key,
    })


duplicate_backup_keys = {
    key: count
    for key, count in backup_key_counts.items()
    if count > 1
}

if duplicate_backup_keys:
    raise RuntimeError(
        "Backup contains duplicate normalized folder/title keys: "
        + repr(duplicate_backup_keys)
    )


current_album_rows_by_key = defaultdict(list)

for album in (inventory_current.get("albums") or {}).values():
    key = _create_album_key(album)

    if key is None:
        continue

    folder_path, normalized_title = key

    if not _create_is_under_target(
        folder_path,
        CREATE_TARGET_ROOT,
    ):
        continue

    current_album_rows_by_key[key].append({
        "current_album_uuid": str(album.get("uuid") or ""),
        "folder_path": folder_path,
        "album_title": str(album.get("title") or "").strip(),
    })


# -----------------------------------------------------------------------
# Select truly missing Backup albums and map every member to Current
# -----------------------------------------------------------------------

review_rows = []
manifest_rows = []
unsafe_rows = []
excluded_rows = []
already_present_rows = []
create_album_jobs = []

backup_album_rows.sort(
    key=lambda row: (
        row["target_folder_path"],
        row["normalized_title"],
        row["backup_album_uuid"],
    )
)

for backup_row in backup_album_rows:
    key = backup_row["key"]
    current_matches = current_album_rows_by_key.get(key, [])

    if (
        backup_row["normalized_title"]
        in CREATE_EXCLUDED_NORMALIZED_TITLES
    ):
        status = "EXCLUDED_SPECIAL_PIN_ALBUM"
        excluded_rows.append(backup_row)

    elif current_matches:
        status = "ALREADY_PRESENT_BY_NORMALIZED_PATH_TITLE"
        already_present_rows.append(backup_row)

    else:
        status = "CREATE_CANDIDATE"

    members = backup_members_by_album_uuid.get(
        backup_row["backup_album_uuid"],
        [],
    )

    mapped_assets = []
    mapping_errors = []

    if status == "CREATE_CANDIDATE":
        if not members:
            mapping_errors.append("BACKUP_ALBUM_HAS_ZERO_ASSETS")

        for backup_asset in members:
            unique_id = backup_asset.get(
                "photo_library_asset_unique_id"
            )

            if unique_id is None:
                mapping_errors.append(
                    "BACKUP_ASSET_WITHOUT_UNIQUE_ID:"
                    + str(backup_asset.get("uuid") or "")
                )
                continue

            candidates = current_asset_index.get(
                tuple(unique_id),
                [],
            )

            if len(candidates) != 1:
                mapping_errors.append(
                    "CURRENT_ASSET_MATCH_COUNT_"
                    + str(len(candidates))
                    + ":"
                    + str(backup_asset.get("uuid") or "")
                )
                continue

            mapped_assets.append({
                "backup_asset": backup_asset,
                "current_asset": candidates[0],
            })

        current_uuids = [
            str(item["current_asset"].get("uuid") or "")
            for item in mapped_assets
        ]

        if len(current_uuids) != len(set(current_uuids)):
            mapping_errors.append(
                "DUPLICATE_CURRENT_UUID_WITHIN_ALBUM"
            )

        if len(mapped_assets) != len(members):
            mapping_errors.append(
                "MAPPED_COUNT_MISMATCH:"
                f"{len(mapped_assets)}!={len(members)}"
            )

        if mapping_errors:
            status = "UNSAFE_ASSET_MAPPING"
            unsafe_rows.append({
                **backup_row,
                "errors": tuple(mapping_errors),
            })
        else:
            create_album_jobs.append({
                **backup_row,
                "mapped_assets": mapped_assets,
                "expected_asset_count": len(mapped_assets),
                "membership_checksum":
                    _create_membership_checksum(
                        current_uuids
                    ),
            })

    review_rows.append({
        "status": status,
        "target_folder_path":
            backup_row["target_folder_path"],
        "album_title":
            backup_row["album_title"],
        "normalized_title":
            backup_row["normalized_title"],
        "backup_album_uuid":
            backup_row["backup_album_uuid"],
        "backup_asset_count": len(members),
        "current_normalized_match_count":
            len(current_matches),
        "current_album_uuids":
            " || ".join(
                row["current_album_uuid"]
                for row in current_matches
            ),
        "errors":
            " || ".join(mapping_errors),
    })


# Never export a partial create plan.
if unsafe_rows:
    print("=" * 100)
    print("CREATE-MISSING-ALBUM export aborted")
    print("=" * 100)
    print("Unsafe album count:", len(unsafe_rows))

    for row in unsafe_rows:
        print()
        print("Album:", row["target_folder_path"], "/", row["album_title"])
        print("Backup UUID:", row["backup_album_uuid"])
        print("Errors:", " || ".join(row["errors"]))

    raise RuntimeError(
        "At least one missing album has unsafe asset mapping. "
        "No create manifest was written."
    )


# -----------------------------------------------------------------------
# Build one manifest row per existing Current asset
# -----------------------------------------------------------------------

for sequence, job in enumerate(
    create_album_jobs,
    start=1,
):
    sorted_assets = sorted(
        job["mapped_assets"],
        key=lambda item: (
            str(
                item["backup_asset"].get("date")
                or ""
            ),
            str(
                item["backup_asset"].get(
                    "original_filename"
                )
                or item["backup_asset"].get("filename")
                or ""
            ),
            str(
                item["current_asset"].get("uuid")
                or ""
            ),
        ),
    )

    for asset_sequence, item in enumerate(
        sorted_assets,
        start=1,
    ):
        backup_asset = item["backup_asset"]
        current_asset = item["current_asset"]

        manifest_rows.append({
            "operation":
                "create_missing_album_add_existing_assets",
            "sequence": sequence,
            "target_folder_path":
                job["target_folder_path"],
            "album_title":
                job["album_title"],
            "normalized_title":
                job["normalized_title"],
            "backup_album_uuid":
                job["backup_album_uuid"],
            "expected_asset_count":
                job["expected_asset_count"],
            "membership_checksum":
                job["membership_checksum"],
            "asset_sequence": asset_sequence,
            "current_uuid":
                str(current_asset.get("uuid") or ""),
            "backup_uuid":
                str(backup_asset.get("uuid") or ""),
            "filename":
                str(
                    backup_asset.get("original_filename")
                    or backup_asset.get("filename")
                    or ""
                ),
            "media":
                "video"
                if backup_asset.get("is_movie")
                else "photo",
            "date":
                str(backup_asset.get("date") or ""),
            "original_size":
                (
                    f"{backup_asset.get('original_width') or backup_asset.get('width') or '-'}"
                    "x"
                    f"{backup_asset.get('original_height') or backup_asset.get('height') or '-'}"
                ),
        })


# -----------------------------------------------------------------------
# Write Latest, review, and immutable archive
# -----------------------------------------------------------------------

CREATE_INBOX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CREATE_ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

manifest_fields = [
    "operation",
    "sequence",
    "target_folder_path",
    "album_title",
    "normalized_title",
    "backup_album_uuid",
    "expected_asset_count",
    "membership_checksum",
    "asset_sequence",
    "current_uuid",
    "backup_uuid",
    "filename",
    "media",
    "date",
    "original_size",
]

review_fields = [
    "status",
    "target_folder_path",
    "album_title",
    "normalized_title",
    "backup_album_uuid",
    "backup_asset_count",
    "current_normalized_match_count",
    "current_album_uuids",
    "errors",
]


def _create_write_tsv(path, fields, rows):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fields,
            delimiter="\t",
            lineterminator="\n",
            extrasaction="raise",
        )
        writer.writeheader()

        for row in rows:
            writer.writerow({
                field: _create_clean_tsv(row.get(field))
                for field in fields
            })

    temporary_path.replace(path)


_create_write_tsv(
    CREATE_REVIEW_LATEST,
    review_fields,
    review_rows,
)

_create_write_tsv(
    CREATE_MANIFEST_LATEST,
    manifest_fields,
    manifest_rows,
)

archive_path = None

if WRITE_CREATE_ARCHIVE:
    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    archive_path = (
        CREATE_ARCHIVE_DIR
        / (
            f"{timestamp}"
            f"__create_missing_albums_"
            f"{len(create_album_jobs)}.tsv"
        )
    )

    shutil.copy2(
        CREATE_MANIFEST_LATEST,
        archive_path,
    )


# -----------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------

status_counts = Counter(
    row["status"]
    for row in review_rows
)

print("=" * 100)
print("CREATE_MISSING_ALBUM_WITH_EXISTING_ASSETS export")
print("=" * 100)
print("Target root:", CREATE_TARGET_ROOT)
print("Backup album objects:", len(backup_album_rows))
print(
    "Excluded normalized titles:",
    sorted(CREATE_EXCLUDED_NORMALIZED_TITLES),
)
print("Status counts:", dict(status_counts))
print("Albums to create:", len(create_album_jobs))
print("Manifest asset rows:", len(manifest_rows))
print("Manifest:", CREATE_MANIFEST_LATEST)
print("Review:", CREATE_REVIEW_LATEST)

if archive_path is not None:
    print("Archive:", archive_path)

print()
print("Create jobs")
print("-" * 100)

for job in create_album_jobs:
    print(
        f"{job['expected_asset_count']:>4} assets | "
        f"{job['target_folder_path']} / "
        f"{job['album_title']}"
    )


create_missing_albums_export_result = {
    "target_root": CREATE_TARGET_ROOT,
    "status_counts": dict(status_counts),
    "review_rows": review_rows,
    "create_album_jobs": create_album_jobs,
    "manifest_rows": manifest_rows,
    "manifest_path": str(CREATE_MANIFEST_LATEST),
    "review_path": str(CREATE_REVIEW_LATEST),
    "archive_path":
        str(archive_path)
        if archive_path is not None
        else None,
}


## REPORT-05 — Backup Album Source Snapshot Under One Folder

**Purpose:** Print a human-readable Backup album list under one selected folder.  
**Requires:** `SETUP-02`.  
**Risk:** Read-only.


In [ ]:
# ============================================================
# Optional helper: Backup album source snapshot under one folder
# ============================================================
#
# Purpose:
#   Print a human-readable list of Backup albums under one selected folder.
#
# This helper:
#   - uses Backup only
#   - does not compare Current
#   - does not verify repair results
#   - does not export a PhotoKit repair manifest
#   - is not used by later cells
#
# Suggested use:
#   Run once, copy the output to a separate text file, then clear the output.
# ============================================================

TARGET_PARENT_FOLDER_PATH = "NSFW"


def optional_album_snapshot_normalize_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = []

    for part in text.split("/"):
        part = part.strip()

        if part:
            parts.append(part)

    return " / ".join(parts)


def optional_album_snapshot_album_folder_path(album):
    folders = album.get("folders") or {}
    paths = []

    for folder in folders.values():
        path = folder.get("path") or folder.get("title")
        path = optional_album_snapshot_normalize_folder_path(path)

        if path:
            paths.append(path)

    if not paths:
        return ""

    return sorted(
        paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def optional_album_snapshot_album_path(album):
    title = album.get("title")

    if not title:
        return None

    folder_path = optional_album_snapshot_album_folder_path(album)

    if folder_path:
        return folder_path + " / " + str(title)

    return str(title)


def optional_album_snapshot_asset_album_paths(asset):
    result = []

    for album in (asset.get("albums") or {}).values():
        path = optional_album_snapshot_album_path(album)

        if path:
            result.append(path)

    return tuple(sorted(set(result)))


def optional_album_snapshot_is_under_target(album_path, target_parent):
    return (
        album_path == target_parent
        or album_path.startswith(target_parent + " / ")
    )


target_parent = optional_album_snapshot_normalize_folder_path(
    TARGET_PARENT_FOLDER_PATH
)

target_album_rows_by_path = {}

for asset in inventory_backup.get("assets") or []:
    for album_path in optional_album_snapshot_asset_album_paths(asset):
        if not optional_album_snapshot_is_under_target(album_path, target_parent):
            continue

        row = target_album_rows_by_path.setdefault(
            album_path,
            {
                "album_path": album_path,
                "assets": 0,
                "photos": 0,
                "videos": 0,
            },
        )

        row["assets"] += 1

        if asset.get("is_movie"):
            row["videos"] += 1
        else:
            row["photos"] += 1

target_album_rows = list(target_album_rows_by_path.values())

target_album_rows.sort(
    key=lambda row: (
        -row["assets"],
        row["album_path"],
    )
)

output_lines = []

output_lines.append("=" * 120)
output_lines.append(
    "Backup album source snapshot under folder: {}".format(target_parent)
)
output_lines.append("=" * 120)
output_lines.append("")
output_lines.append("Purpose:")
output_lines.append("  Backup-only album source list for manual reference.")
output_lines.append("  It does not compare Current.")
output_lines.append("  It does not verify repair results.")
output_lines.append("  It does not export a PhotoKit repair manifest.")
output_lines.append("")
output_lines.append("backup album count: {}".format(len(target_album_rows)))
output_lines.append("")
output_lines.append(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "idx",
        "backup_assets",
        "backup_photos",
        "backup_videos",
        "backup_album_path",
    )
)
output_lines.append(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "---",
        "-------------",
        "-------------",
        "-------------",
        "-" * 80,
    )
)

for index, row in enumerate(target_album_rows, start=1):
    output_lines.append(
        "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
            index,
            row["assets"],
            row["photos"],
            row["videos"],
            row["album_path"],
        )
    )

print("\n".join(output_lines))

## REPORT-06 — Backup vs Current Paths and Asset Counts

**Purpose:** Compare Backup and Current album paths and counts across the whole library or one selected root folder.  
**Requires:** `SETUP-02`.  
**Risk:** Read-only.


In [ ]:
# =====================================================================
# Check: Backup vs Current album paths and asset counts, whole library
# =====================================================================

from collections import defaultdict, Counter
import hashlib


# None = whole library.
# Or set to "NSFW", "股票", etc. to limit to one root folder.
TARGET_ROOT_FOLDER_PATH = None


def _albumcmp_normalize_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]

    return " / ".join(parts)


def _albumcmp_join_photo_path(folder_path, album_title):
    folder = _albumcmp_normalize_folder_path(folder_path)
    album = str(album_title).strip()

    if folder:
        return f"{folder} / {album}"

    return album


def _albumcmp_deepest_folder_path_for_album(album):
    folders = album.get("folders") or {}

    folder_paths = [
        _albumcmp_normalize_folder_path(folder.get("path") or folder.get("title"))
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    ]

    folder_paths = [
        path
        for path in folder_paths
        if path
    ]

    if not folder_paths:
        return ""

    return sorted(
        folder_paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def _albumcmp_album_full_path(album):
    album_title = album.get("title")

    if not album_title:
        return None

    folder_path = _albumcmp_deepest_folder_path_for_album(album)

    return _albumcmp_join_photo_path(
        folder_path,
        str(album_title).strip(),
    )


def _albumcmp_is_under_root(album_path, root_folder_path):
    if root_folder_path is None:
        return True

    root = _albumcmp_normalize_folder_path(root_folder_path)

    if not root:
        return True

    return (
        album_path == root
        or album_path.startswith(root + " / ")
    )


def _albumcmp_asset_key(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is not None:
        return "unique_id:" + repr(tuple(unique_id))

    uuid = asset.get("uuid")

    if uuid:
        return "uuid:" + str(uuid)

    return None


def _albumcmp_checksum(member_set):
    joined = "\n".join(sorted(member_set))
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:16]


def _albumcmp_root_folder_from_album_path(album_path):
    parts = [
        part.strip()
        for part in str(album_path or "").split(" / ")
        if part.strip()
    ]

    if len(parts) <= 1:
        return "[ROOT_ALBUMS]"

    return parts[0]


def _albumcmp_collect_album_catalog(inventory, root_folder_path=None):
    album_rows = []
    album_path_counts = Counter()

    for album in (inventory.get("albums") or {}).values():
        album_path = _albumcmp_album_full_path(album)

        if not album_path:
            continue

        if not _albumcmp_is_under_root(album_path, root_folder_path):
            continue

        folder_path = _albumcmp_deepest_folder_path_for_album(album)
        album_title = str(album.get("title") or "").strip()

        album_path_counts[album_path] += 1

        album_rows.append({
            "folder_path": folder_path,
            "album_title": album_title,
            "album_path": album_path,
            "root_folder": _albumcmp_root_folder_from_album_path(album_path),
            "album_uuid": album.get("uuid") or "-",
        })

    album_rows.sort(
        key=lambda row: (
            row["album_path"],
            row["album_uuid"],
        )
    )

    return album_rows, album_path_counts


def _albumcmp_collect_asset_members_by_album_path(inventory, root_folder_path=None):
    members_by_album_path = defaultdict(set)
    no_album_asset_keys = set()

    for asset in inventory.get("assets") or []:
        asset_key = _albumcmp_asset_key(asset)

        if asset_key is None:
            continue

        asset_albums = asset.get("albums") or {}

        if not asset_albums:
            no_album_asset_keys.add(asset_key)
            continue

        for album in asset_albums.values():
            album_path = _albumcmp_album_full_path(album)

            if not album_path:
                continue

            if not _albumcmp_is_under_root(album_path, root_folder_path):
                continue

            members_by_album_path[album_path].add(asset_key)

    return members_by_album_path, no_album_asset_keys


backup_album_catalog, backup_album_path_counts = _albumcmp_collect_album_catalog(
    inventory_backup,
    TARGET_ROOT_FOLDER_PATH,
)

current_album_catalog, current_album_path_counts = _albumcmp_collect_album_catalog(
    inventory_current,
    TARGET_ROOT_FOLDER_PATH,
)

backup_members_by_album_path, backup_no_album_assets = _albumcmp_collect_asset_members_by_album_path(
    inventory_backup,
    TARGET_ROOT_FOLDER_PATH,
)

current_members_by_album_path, current_no_album_assets = _albumcmp_collect_asset_members_by_album_path(
    inventory_current,
    TARGET_ROOT_FOLDER_PATH,
)

backup_album_paths = set(row["album_path"] for row in backup_album_catalog)
current_album_paths = set(row["album_path"] for row in current_album_catalog)

all_album_paths = sorted(backup_album_paths | current_album_paths)

comparison_rows = []

for album_path in all_album_paths:
    backup_members = backup_members_by_album_path.get(album_path, set())
    current_members = current_members_by_album_path.get(album_path, set())

    missing_members = backup_members - current_members
    extra_members = current_members - backup_members

    backup_album_exists = album_path in backup_album_paths
    current_album_exists = album_path in current_album_paths

    if backup_album_exists and not current_album_exists:
        status = "MISSING_ALBUM_IN_CURRENT"
    elif current_album_exists and not backup_album_exists:
        status = "CURRENT_ONLY_ALBUM"
    elif not missing_members and not extra_members:
        status = "OK"
    else:
        status = "MEMBERSHIP_DIFF"

    comparison_rows.append({
        "status": status,
        "root_folder": _albumcmp_root_folder_from_album_path(album_path),
        "album_path": album_path,
        "backup_album_objects": backup_album_path_counts.get(album_path, 0),
        "current_album_objects": current_album_path_counts.get(album_path, 0),
        "backup_assets": len(backup_members),
        "current_assets": len(current_members),
        "missing_assets_in_current": len(missing_members),
        "extra_assets_in_current": len(extra_members),
        "backup_checksum": _albumcmp_checksum(backup_members),
        "current_checksum": _albumcmp_checksum(current_members),
        "checksum_match": _albumcmp_checksum(backup_members) == _albumcmp_checksum(current_members),
    })


target_label = (
    "whole library"
    if TARGET_ROOT_FOLDER_PATH is None
    else _albumcmp_normalize_folder_path(TARGET_ROOT_FOLDER_PATH)
)

print("=" * 180)
print("Backup vs Current album paths and asset counts:", target_label)
print("=" * 180)
print()

print("album catalog counts")
print("-" * 180)
print("backup album objects:       ", len(backup_album_catalog))
print("current album objects:      ", len(current_album_catalog))
print("backup unique album paths:  ", len(backup_album_paths))
print("current unique album paths: ", len(current_album_paths))
print("all unique album paths:     ", len(all_album_paths))
print()

print("no-album asset counts")
print("-" * 180)
print("backup no-album assets: ", len(backup_no_album_assets))
print("current no-album assets:", len(current_no_album_assets))
print()

status_counts = Counter(row["status"] for row in comparison_rows)

print("status counts")
print("-" * 180)

for status, count in sorted(status_counts.items()):
    print(f"{status}: {count}")

print()
print("root folder breakdown")
print("-" * 180)
print(
    "{:<50} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
        "root_folder",
        "backup",
        "current",
        "missing",
        "new",
        "diff",
    )
)

all_root_folders = sorted(
    set(row["root_folder"] for row in comparison_rows)
)

for root_folder in all_root_folders:
    rows = [
        row
        for row in comparison_rows
        if row["root_folder"] == root_folder
    ]

    backup_count = sum(1 for row in rows if row["backup_album_objects"] > 0)
    current_count = sum(1 for row in rows if row["current_album_objects"] > 0)
    missing_count = sum(1 for row in rows if row["status"] == "MISSING_ALBUM_IN_CURRENT")
    new_count = sum(1 for row in rows if row["status"] == "CURRENT_ONLY_ALBUM")
    diff_count = sum(1 for row in rows if row["status"] == "MEMBERSHIP_DIFF")

    print(
        "{:<50} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
            root_folder[:50],
            backup_count,
            current_count,
            missing_count,
            new_count,
            diff_count,
        )
    )

print()
print("missing / current-only album paths")
print("-" * 180)

for row in comparison_rows:
    if row["status"] in {"MISSING_ALBUM_IN_CURRENT", "CURRENT_ONLY_ALBUM"}:
        print(
            f"{row['status']} | "
            f"root={row['root_folder']} | "
            f"backup_album_objects={row['backup_album_objects']} | "
            f"current_album_objects={row['current_album_objects']} | "
            f"backup_assets={row['backup_assets']} | "
            f"current_assets={row['current_assets']} | "
            f"{row['album_path']}"
        )

print()
print("membership differences")
print("-" * 180)

for row in comparison_rows:
    if row["status"] == "MEMBERSHIP_DIFF":
        print(
            f"{row['status']} | "
            f"root={row['root_folder']} | "
            f"backup_assets={row['backup_assets']} | "
            f"current_assets={row['current_assets']} | "
            f"missing_assets_in_current={row['missing_assets_in_current']} | "
            f"extra_assets_in_current={row['extra_assets_in_current']} | "
            f"{row['album_path']}"
        )

print()
print("duplicate full album paths")
print("-" * 180)

duplicate_found = False

for album_path, count in sorted(backup_album_path_counts.items()):
    if count > 1:
        duplicate_found = True
        print(f"BACKUP duplicate path x{count}: {album_path}")

for album_path, count in sorted(current_album_path_counts.items()):
    if count > 1:
        duplicate_found = True
        print(f"CURRENT duplicate path x{count}: {album_path}")

if not duplicate_found:
    print("-")

print()
print("full table")
print("-" * 180)
print(
    "{:<24} {:<28} {:>7} {:>7} {:>8} {:>8} {:>8} {:>8} {:>18} {:>18}  {}".format(
        "status",
        "root_folder",
        "b_alb",
        "c_alb",
        "b_assets",
        "c_assets",
        "missing",
        "extra",
        "backup_checksum",
        "current_checksum",
        "album_path",
    )
)

for row in comparison_rows:
    print(
        "{:<24} {:<28} {:>7} {:>7} {:>8} {:>8} {:>8} {:>8} {:>18} {:>18}  {}".format(
            row["status"],
            row["root_folder"][:28],
            row["backup_album_objects"],
            row["current_album_objects"],
            row["backup_assets"],
            row["current_assets"],
            row["missing_assets_in_current"],
            row["extra_assets_in_current"],
            row["backup_checksum"],
            row["current_checksum"],
            row["album_path"],
        )
    )

## REPORT-07 — Duplicate Album Path Review

**Purpose:** Find duplicate album paths in Backup, show the corresponding albums in Current, and compare the merged asset memberships across both libraries.

**Requires:** `SETUP-02`.

**Scope:** Uses duplicate album paths found in Backup as the review universe. For each path, all same-path albums are conceptually merged within each library before comparing cross-library asset identities.

**Risk:** Analysis only. Prints results in the Notebook; does not modify either Photos Library and does not write report files.

In [ ]:
# =====================================================================
# REPORT-07 — Duplicate Album Path Review
#
# Backup-centric universe:
# - Find full album paths represented by at least 2 albums in Backup.
# - Show all same-path albums in Backup and Current.
# - Conceptually merge their memberships within each library.
# - Compare the merged cross-library asset unique-ID sets.
#
# Read-only:
# - Does not modify either Photos Library.
# - Does not write TSV / JSON files.
# =====================================================================

from collections import defaultdict


def _report07_normalize_folder_path(path):
    """
    Normalize a folder path for comparison and display.

    Examples:
        "HIDE/Girls"       -> "HIDE / Girls"
        "HIDE / Girls"     -> "HIDE / Girls"
        " HIDE/Girls "     -> "HIDE / Girls"
    """
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")

    parts = [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]

    return " / ".join(parts)


def _report07_deepest_folder_path_for_album(album):
    """
    Return the deepest folder path attached to one album.

    An album stores the full folder chain in album["folders"].
    The deepest path represents the album's actual containing folder.
    """
    folders = album.get("folders") or {}

    folder_paths = []

    for folder in folders.values():
        raw_path = folder.get("path") or folder.get("title")

        if not raw_path:
            continue

        normalized_path = _report07_normalize_folder_path(raw_path)

        if normalized_path:
            folder_paths.append(normalized_path)

    if not folder_paths:
        return ""

    return sorted(
        folder_paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def _report07_album_full_path(album):
    """
    Build the canonical full album path:

        folder path + album title

    Root-level albums use only the album title.
    """
    album_title = str(album.get("title") or "").strip()

    if not album_title:
        return None

    folder_path = _report07_deepest_folder_path_for_album(album)

    if folder_path:
        return f"{folder_path} / {album_title}"

    return album_title


def _report07_normalize_filename_extension(filename):
    """
    Preserve the filename stem exactly, but normalize extension case.

    Examples:
        IMG_0244.JPG  -> IMG_0244.jpg
        IMG_0244.JPEG -> IMG_0244.jpeg
    """
    if filename is None:
        return None

    text = str(filename)

    if "." not in text:
        return text

    stem, extension = text.rsplit(".", 1)

    if not stem or not extension:
        return text

    return f"{stem}.{extension.lower()}"


def _report07_asset_compare_key(asset):
    """
    Return the normalized cross-library asset identity key.

    Uses the existing project photo_library_asset_unique_id:

        normalized original filename
        date
        file size
        adjustment signature
        optional SHA256 collision discriminator

    Album UUID and asset UUID are intentionally not used across libraries.
    """
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        raise RuntimeError(
            "REPORT-07 found a normal asset without "
            "photo_library_asset_unique_id:\n"
            f"{asset}"
        )

    normalized = list(tuple(unique_id))

    if normalized:
        normalized[0] = _report07_normalize_filename_extension(
            normalized[0]
        )

    return tuple(normalized)


def _report07_collect_library_state(inventory):
    """
    Collect:

    1. albums_by_path
       full album path -> list of albums

    2. member_keys_by_album_uuid
       album UUID -> set of normalized asset identity keys

    3. member_row_count_by_album_uuid
       album UUID -> number of membership rows

    The row count and unique-key count should normally be the same inside
    one album, but both are retained so the report does not silently assume it.
    """
    albums_by_path = defaultdict(list)
    member_keys_by_album_uuid = defaultdict(set)
    member_row_count_by_album_uuid = defaultdict(int)

    albums = inventory.get("albums") or {}

    for album in albums.values():
        album_path = _report07_album_full_path(album)

        if album_path is None:
            continue

        albums_by_path[album_path].append(album)

    for asset in inventory.get("assets") or []:
        asset_key = _report07_asset_compare_key(asset)
        asset_albums = asset.get("albums") or {}

        for album_uuid, album in asset_albums.items():
            album_path = _report07_album_full_path(album)

            if album_path is None:
                continue

            member_row_count_by_album_uuid[album_uuid] += 1
            member_keys_by_album_uuid[album_uuid].add(asset_key)

    for album_path in albums_by_path:
        albums_by_path[album_path].sort(
            key=lambda album: (
                -len(
                    member_keys_by_album_uuid.get(
                        album.get("uuid"),
                        set(),
                    )
                ),
                str(album.get("uuid") or ""),
            )
        )

    return {
        "albums_by_path": dict(albums_by_path),
        "member_keys_by_album_uuid": dict(member_keys_by_album_uuid),
        "member_row_count_by_album_uuid": dict(
            member_row_count_by_album_uuid
        ),
    }


def _report07_summarize_album_path(library_state, album_path):
    """
    Conceptually merge all albums at one full album path.

    Returns:
    - album count
    - per-album asset counts
    - total membership rows
    - unique assets after merge
    - duplicate memberships
    - merged unique asset identity set
    """
    albums = (
        library_state["albums_by_path"].get(album_path)
        or []
    )

    member_keys_by_album_uuid = (
        library_state["member_keys_by_album_uuid"]
    )

    member_row_count_by_album_uuid = (
        library_state["member_row_count_by_album_uuid"]
    )

    per_album_rows = []
    merged_unique_keys = set()
    total_membership_rows = 0

    for album in albums:
        album_uuid = album.get("uuid")

        member_keys = set(
            member_keys_by_album_uuid.get(
                album_uuid,
                set(),
            )
        )

        membership_rows = int(
            member_row_count_by_album_uuid.get(
                album_uuid,
                0,
            )
        )

        merged_unique_keys.update(member_keys)
        total_membership_rows += membership_rows

        per_album_rows.append({
            "album_uuid": album_uuid or "-",
            "asset_count": len(member_keys),
            "membership_rows": membership_rows,
        })

    per_album_rows.sort(
        key=lambda row: (
            -row["asset_count"],
            row["album_uuid"],
        )
    )

    duplicate_memberships = (
        total_membership_rows
        - len(merged_unique_keys)
    )

    return {
        "album_count": len(albums),
        "per_album_rows": per_album_rows,
        "total_membership_rows": total_membership_rows,
        "unique_assets_after_merge": len(merged_unique_keys),
        "duplicate_memberships": duplicate_memberships,
        "merged_unique_keys": merged_unique_keys,
    }


def _report07_format_counts(per_album_rows):
    """
    Format per-album asset counts as:

        100, 82, 67, 54
    """
    if not per_album_rows:
        return "-"

    return ", ".join(
        str(row["asset_count"])
        for row in per_album_rows
    )


def _report07_print_library_section(label, summary):
    print(label)
    print(f"Albums: {summary['album_count']}")
    print(
        "Assets in each album:",
        _report07_format_counts(
            summary["per_album_rows"]
        ),
    )
    print(
        "Total membership rows:",
        summary["total_membership_rows"],
    )
    print(
        "Unique assets after merge:",
        summary["unique_assets_after_merge"],
    )
    print(
        "Duplicate memberships:",
        summary["duplicate_memberships"],
    )


backup_state_report07 = _report07_collect_library_state(
    inventory_backup
)

current_state_report07 = _report07_collect_library_state(
    inventory_current
)


backup_duplicate_album_paths_report07 = []

for album_path, albums in (
    backup_state_report07["albums_by_path"].items()
):
    if len(albums) < 2:
        continue

    backup_summary = _report07_summarize_album_path(
        backup_state_report07,
        album_path,
    )

    backup_duplicate_album_paths_report07.append({
        "album_path": album_path,
        "backup_album_count": backup_summary["album_count"],
        "backup_total_membership_rows":
            backup_summary["total_membership_rows"],
    })


backup_duplicate_album_paths_report07.sort(
    key=lambda row: (
        -row["backup_album_count"],
        -row["backup_total_membership_rows"],
        row["album_path"],
    )
)


print("=" * 120)
print("REPORT-07 — Duplicate Album Path Review")
print("=" * 120)
print()
print(
    "Backup duplicate album paths:",
    len(backup_duplicate_album_paths_report07),
)
print(
    "Comparison identity:",
    "normalized photo_library_asset_unique_id",
)
print()


for path_index, path_row in enumerate(
    backup_duplicate_album_paths_report07,
    start=1,
):
    album_path = path_row["album_path"]

    backup_summary = _report07_summarize_album_path(
        backup_state_report07,
        album_path,
    )

    current_summary = _report07_summarize_album_path(
        current_state_report07,
        album_path,
    )

    backup_keys = backup_summary["merged_unique_keys"]
    current_keys = current_summary["merged_unique_keys"]

    shared_keys = backup_keys & current_keys
    backup_missing_from_current = backup_keys - current_keys
    current_only_keys = current_keys - backup_keys

    merged_sets_identical = (
        backup_keys == current_keys
    )

    backup_preservation_complete = (
        len(backup_missing_from_current) == 0
    )

    print("=" * 120)
    print(
        f"[{path_index}/"
        f"{len(backup_duplicate_album_paths_report07)}] "
        f"Album Path: {album_path}"
    )
    print("=" * 120)
    print()

    _report07_print_library_section(
        "BACKUP",
        backup_summary,
    )

    print()

    _report07_print_library_section(
        "CURRENT",
        current_summary,
    )

    print()
    print("CROSS-LIBRARY MERGED MEMBERSHIP")
    print(
        "Backup unique assets:",
        len(backup_keys),
    )
    print(
        "Current unique assets:",
        len(current_keys),
    )
    print(
        "Shared unique assets:",
        len(shared_keys),
    )
    print(
        "Backup assets found in Current:",
        len(shared_keys),
    )
    print(
        "Backup assets missing from Current:",
        len(backup_missing_from_current),
    )
    print(
        "Current-only assets:",
        len(current_only_keys),
    )
    print(
        "Merged unique-ID sets identical:",
        "YES" if merged_sets_identical else "NO",
    )
    print(
        "Backup preservation complete:",
        "YES" if backup_preservation_complete else "NO",
    )

    print()


if not backup_duplicate_album_paths_report07:
    print("No duplicate album paths were found in Backup.")

## ARCHIVE-01 — TEMP Verify Job 1 Duplicate Deletion

**Purpose:** Historical one-job live verification.  
**Status:** Temporary diagnostic retained for reference.  
**Risk:** Read-only, but not part of the normal workflow.


In [ ]:
# ============================================================
# TEMP — Verify Job 1 duplicate deletion
#
# READ ONLY:
# - reads the live Current Default Photos Library database
# - does not use inventory_current cache
# - does not modify Photos Library
# ============================================================

from pathlib import Path
import json
import osxphotos


OLD_ROOT_ALBUM_UUID = "C0F17467-F7FD-4346-8F99-121D7DD1E9DD"
NEW_STOCK_ALBUM_UUID = "205EB2D8-3689-41F5-B6D7-7E8506505B38"
EXPECTED_ASSET_COUNT = 168


# Read the saved Current Default Photos Library path.
history_path = Path(
    "data/local_config/test2_library_paths.json"
)

library_history = json.loads(
    history_path.read_text(encoding="utf-8")
)

current_library_path = Path(
    library_history["current_default"]
)

print("Current Library:", current_library_path)
print()


# Read current album objects directly from the live database.
photosdb = osxphotos.PhotosDB(
    str(current_library_path)
)

albums_by_uuid = {
    str(album.uuid): album
    for album in photosdb.album_info
}


def inspect_album(label, album_uuid):
    album = albums_by_uuid.get(album_uuid)

    print("=" * 80)
    print(label)
    print("=" * 80)

    if album is None:
        print("Album exists: NO")
        print("UUID:", album_uuid)
        print()
        return None

    folder_names = list(
        album.folder_names or []
    )

    folder_path = (
        " / ".join(folder_names)
        if folder_names
        else "[ROOT_ALBUMS]"
    )

    print("Album exists: YES")
    print("UUID:", album.uuid)
    print("Folder Path:", folder_path)
    print("Asset Count:", len(album.photos))
    print("Title Length:", len(album.title))
    print("Title Ending:", repr(album.title[-40:]))
    print()

    return {
        "folder_path": folder_path,
        "asset_count": len(album.photos),
        "title": album.title,
    }


old_root = inspect_album(
    "OLD ROOT ALBUM",
    OLD_ROOT_ALBUM_UUID,
)

new_stock = inspect_album(
    "NEW 股票 ALBUM",
    NEW_STOCK_ALBUM_UUID,
)


root_deleted_ok = old_root is None

stock_exists_ok = new_stock is not None

stock_path_ok = (
    new_stock is not None
    and new_stock["folder_path"] == "股票"
)

stock_count_ok = (
    new_stock is not None
    and new_stock["asset_count"] == EXPECTED_ASSET_COUNT
)


print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)
print(
    "Root duplicate deleted:",
    "PASS" if root_deleted_ok else "FAIL",
)
print(
    "股票 album remains:",
    "PASS" if stock_exists_ok else "FAIL",
)
print(
    "股票 folder path correct:",
    "PASS" if stock_path_ok else "FAIL",
)
print(
    f"股票 asset count is {EXPECTED_ASSET_COUNT}:",
    "PASS" if stock_count_ok else "FAIL",
)
print()

all_ok = (
    root_deleted_ok
    and stock_exists_ok
    and stock_path_ok
    and stock_count_ok
)

print(
    "OVERALL RESULT:",
    "SUCCESS" if all_ok else "NOT YET CORRECT",
)

## ARCHIVE-02 — OLD Duplicate Diagnostic Archive

**Purpose:** Historical duplicate diagnostic code.  
**Status:** Old/archive; do not use in the normal workflow.


In [ ]:
# # ============================================================
# # Appendix A: Duplicate diagnostic archive
# # 
# # TEMP: Diagnose potential duplicate groups by SHA256
# #       with full manual-review metadata
# # ============================================================

# import hashlib
# import os
# import time
# from datetime import datetime


# def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
#     cached_sha256 = asset.get("content_sha256")
#     if cached_sha256:
#         return cached_sha256

#     path = asset.get("path")

#     if path is None:
#         return None

#     if not os.path.exists(path):
#         return None

#     sha256 = hashlib.sha256()

#     with open(path, "rb") as f:
#         while True:
#             chunk = f.read(chunk_size)

#             if not chunk:
#                 break

#             sha256.update(chunk)

#     digest = sha256.hexdigest()
#     asset["content_sha256"] = digest
#     return digest


# def build_potential_duplicate_groups_by_unique_id(inventory):
#     unique_id_to_assets = {}

#     for asset in inventory["assets"]:
#         unique_id = asset.get("photo_library_asset_unique_id")

#         if unique_id is None:
#             continue

#         if unique_id not in unique_id_to_assets:
#             unique_id_to_assets[unique_id] = []

#         unique_id_to_assets[unique_id].append(asset)

#     return {
#         unique_id: assets
#         for unique_id, assets in unique_id_to_assets.items()
#         if len(assets) > 1
#     }


# def normalize_string_list(values):
#     result = []

#     if values is None:
#         return result

#     if isinstance(values, str):
#         return [values]

#     if isinstance(values, dict):
#         iterable = values.values()
#     elif isinstance(values, (list, tuple, set)):
#         iterable = values
#     else:
#         return [str(values)]

#     for item in iterable:
#         if item is None:
#             continue

#         if isinstance(item, str):
#             value = item
#         elif isinstance(item, dict):
#             value = (
#                 item.get("title")
#                 or item.get("name")
#                 or item.get("path")
#                 or item.get("folder_path")
#                 or item.get("album_path")
#             )
#         else:
#             value = str(item)

#         if value:
#             result.append(value)

#     return sorted(set(result))


# def get_asset_album_titles(asset):
#     albums = asset.get("albums")
#     return normalize_string_list(albums)


# def get_asset_folder_paths(asset):
#     folders = asset.get("folders")
#     folder_paths = normalize_string_list(folders)

#     # Some inventory formats may store folder paths under different keys.
#     extra_candidates = [
#         asset.get("folder_paths"),
#         asset.get("folder_path"),
#         asset.get("album_folder_paths"),
#     ]

#     for candidate in extra_candidates:
#         folder_paths.extend(normalize_string_list(candidate))

#     return sorted(set(folder_paths))


# def get_asset_keywords(asset):
#     keywords = asset.get("keywords")
#     return normalize_string_list(keywords)


# def get_asset_description(asset):
#     return (
#         asset.get("description")
#         or asset.get("caption")
#         or asset.get("title")
#         or ""
#     )


# def parse_date_added_for_sort(asset):
#     date_added = asset.get("date_added")

#     if not date_added:
#         return datetime.max

#     if isinstance(date_added, datetime):
#         return date_added

#     text = str(date_added)

#     try:
#         return datetime.fromisoformat(text.replace("Z", "+00:00"))
#     except Exception:
#         return datetime.max


# def asset_metadata_signature(asset):
#     return {
#         "albums": tuple(get_asset_album_titles(asset)),
#         "folders": tuple(get_asset_folder_paths(asset)),
#         "keywords": tuple(get_asset_keywords(asset)),
#         "description": get_asset_description(asset),
#         "favorite": asset.get("favorite"),
#         "hidden": asset.get("hidden"),
#         "hasadjustments": asset.get("hasadjustments"),
#         "adjustment_signature": asset.get("adjustment_signature"),
#     }


# def metadata_score(asset):
#     return (
#         len(get_asset_album_titles(asset)) * 10
#         + len(get_asset_folder_paths(asset)) * 10
#         + len(get_asset_keywords(asset)) * 5
#         + (1 if get_asset_description(asset) else 0)
#         + (1 if asset.get("favorite") else 0)
#         + (1 if asset.get("hidden") else 0)
#     )


# def choose_representative_asset(assets):
#     # Prefer metadata-rich assets; tie-break by earliest Date Added.
#     return sorted(
#         assets,
#         key=lambda asset: (
#             -metadata_score(asset),
#             parse_date_added_for_sort(asset),
#             asset.get("uuid") or "",
#         ),
#     )[0]


# def print_asset_manual_review_block(asset, indent="  "):
#     print(f"{indent}UUID:", asset.get("uuid"))
#     print(f"{indent}Original File Name:", asset.get("original_filename"))
#     print(f"{indent}Filename:", asset.get("filename"))
#     print(f"{indent}Date:", asset.get("date"))
#     print(f"{indent}Date Added:", asset.get("date_added"))
#     print(f"{indent}File Size:", asset.get("file_size_bytes"))
#     print(f"{indent}Has Adjustments:", asset.get("hasadjustments"))
#     print(f"{indent}Adjustment Signature:", asset.get("adjustment_signature"))
#     print(f"{indent}Width x Height:", asset.get("width"), "x", asset.get("height"))
#     print(f"{indent}Original Width x Height:", asset.get("original_width"), "x", asset.get("original_height"))
#     print(f"{indent}Albums:", get_asset_album_titles(asset))
#     print(f"{indent}Folder Paths:", get_asset_folder_paths(asset))
#     print(f"{indent}Keywords:", get_asset_keywords(asset))
#     print(f"{indent}Description:", get_asset_description(asset))
#     print(f"{indent}Favorite:", asset.get("favorite"))
#     print(f"{indent}Hidden:", asset.get("hidden"))
#     print(f"{indent}Path:", asset.get("path"))


# def print_cleanup_recommendation(assets):
#     metadata_signatures = [asset_metadata_signature(asset) for asset in assets]
#     metadata_all_same = all(
#         signature == metadata_signatures[0]
#         for signature in metadata_signatures
#     )

#     representative = choose_representative_asset(assets)

#     if metadata_all_same:
#         print("Recommendation:")
#         print("  Metadata appears identical.")
#         print("  Keep earliest / representative asset:")
#         print("   ", representative.get("uuid"))
#         print("  Delete other duplicate asset(s):")
#         for asset in assets:
#             if asset is not representative:
#                 print("   ", asset.get("uuid"))
#     else:
#         print("Recommendation:")
#         print("  Metadata differs across duplicate assets.")
#         print("  Do NOT blindly delete.")
#         print("  Suggested representative, based on richer metadata + earliest Date Added:")
#         print("   ", representative.get("uuid"))
#         print("  Before deleting others, manually confirm whether album/folder/keyword membership should be preserved.")


# def diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory,
#     label,
#     max_true_duplicate_groups_to_print=50,
#     max_key_collision_groups_to_print=20,
# ):
#     start_time = time.perf_counter()

#     potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

#     true_content_duplicate_groups = []
#     key_collision_groups = []
#     sha_error_assets = []

#     checked_asset_count = 0

#     for unique_id, assets in potential_groups.items():
#         sha256_to_assets = {}

#         for asset in assets:
#             checked_asset_count += 1
#             sha256 = compute_sha256_for_asset(asset)

#             if sha256 is None:
#                 sha_error_assets.append(asset)
#                 continue

#             if sha256 not in sha256_to_assets:
#                 sha256_to_assets[sha256] = []

#             sha256_to_assets[sha256].append(asset)

#         duplicate_sha_groups = {
#             sha256: sha_assets
#             for sha256, sha_assets in sha256_to_assets.items()
#             if len(sha_assets) > 1
#         }

#         if duplicate_sha_groups:
#             for sha256, sha_assets in duplicate_sha_groups.items():
#                 true_content_duplicate_groups.append(
#                     {
#                         "unique_id": unique_id,
#                         "sha256": sha256,
#                         "assets": sha_assets,
#                     }
#                 )

#         if len(sha256_to_assets) > 1:
#             key_collision_groups.append(
#                 {
#                     "unique_id": unique_id,
#                     "sha256_to_assets": sha256_to_assets,
#                 }
#             )

#     elapsed = time.perf_counter() - start_time

#     print(label)
#     print("-" * 120)
#     print("potential duplicate unique_id group count:", len(potential_groups))
#     print("checked asset count:", checked_asset_count)
#     print("sha error asset count:", len(sha_error_assets))
#     print("true content duplicate group count:", len(true_content_duplicate_groups))
#     print("key collision group count:", len(key_collision_groups))
#     print("elapsed seconds:", round(elapsed, 3))

#     print()
#     print("TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW")
#     print("-" * 120)

#     for index, group in enumerate(true_content_duplicate_groups, start=1):
#         if index > max_true_duplicate_groups_to_print:
#             print("... more true content duplicate groups not printed")
#             break

#         assets_sorted = sorted(
#             group["assets"],
#             key=lambda asset: (
#                 parse_date_added_for_sort(asset),
#                 asset.get("uuid") or "",
#             ),
#         )

#         print("=" * 120)
#         print(f"Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256:", group["sha256"])
#         print("asset count:", len(assets_sorted))

#         first_asset = assets_sorted[0]
#         print("Original File Name:", first_asset.get("original_filename"))
#         print("Date:", first_asset.get("date"))
#         print("File Size:", first_asset.get("file_size_bytes"))
#         print("Adjustment Signature:", first_asset.get("adjustment_signature"))

#         union_albums = sorted(
#             set(
#                 album
#                 for asset in assets_sorted
#                 for album in get_asset_album_titles(asset)
#             )
#         )
#         union_folders = sorted(
#             set(
#                 folder
#                 for asset in assets_sorted
#                 for folder in get_asset_folder_paths(asset)
#             )
#         )
#         union_keywords = sorted(
#             set(
#                 keyword
#                 for asset in assets_sorted
#                 for keyword in get_asset_keywords(asset)
#             )
#         )

#         print("Union Albums:", union_albums)
#         print("Union Folder Paths:", union_folders)
#         print("Union Keywords:", union_keywords)

#         print()
#         print_cleanup_recommendation(assets_sorted)
#         print()

#         for asset_index, asset in enumerate(assets_sorted, start=1):
#             print("-" * 120)
#             print(f"Asset {asset_index}")
#             print_asset_manual_review_block(asset, indent="  ")

#         print()

#     print()
#     print("KEY COLLISION GROUPS")
#     print("-" * 120)

#     for index, group in enumerate(key_collision_groups, start=1):
#         if index > max_key_collision_groups_to_print:
#             print("... more key collision groups not printed")
#             break

#         print("=" * 120)
#         print(f"Key Collision Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256 count:", len(group["sha256_to_assets"]))

#         for sha256, assets in group["sha256_to_assets"].items():
#             print("  sha256:", sha256)
#             print("  asset count:", len(assets))

#             for asset in assets:
#                 print("    uuid:", asset.get("uuid"))
#                 print("    original_filename:", asset.get("original_filename"))
#                 print("    filename:", asset.get("filename"))
#                 print("    date:", asset.get("date"))
#                 print("    date_added:", asset.get("date_added"))
#                 print("    file_size_bytes:", asset.get("file_size_bytes"))
#                 print("    albums:", get_asset_album_titles(asset))
#                 print("    folder_paths:", get_asset_folder_paths(asset))
#                 print("    keywords:", get_asset_keywords(asset))
#                 print("    path:", asset.get("path"))

#         print()

#     return {
#         "potential_groups": potential_groups,
#         "true_content_duplicate_groups": true_content_duplicate_groups,
#         "key_collision_groups": key_collision_groups,
#         "sha_error_assets": sha_error_assets,
#     }


# backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_backup,
#     "BACKUP potential duplicate diagnostic with metadata",
# )

# print()

# current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_current,
#     "CURRENT potential duplicate diagnostic with metadata",
# )

## ARCHIVE-03 — OLD Debug Assets Without Unique ID

**Purpose:** Historical debug dump for assets without unique IDs.  
**Status:** Old/archive; run only for that specific debugging need.


In [ ]:
# # ============================================================
# # Appendix B: Debug assets without unique ID
# # 
# # DEBUG: Dump assets without photo_library_asset_unique_id
# # ============================================================

# import os
# import time
# from collections import Counter

# def debug_dump_assets_without_photo_library_asset_unique_id(inventory, label, max_print=80):
#     missing_assets = [
#         asset
#         for asset in inventory["assets"]
#         if asset.get("photo_library_asset_unique_id") is None
#     ]

#     reason_counter = Counter()

#     print(label)
#     print("-" * 120)
#     print("assets without photo_library_asset_unique_id:", len(missing_assets))
#     print()

#     for asset in missing_assets:
#         path = asset.get("path")
#         original_filename = asset.get("original_filename")
#         filename = asset.get("filename")
#         date = asset.get("date")
#         file_size_bytes = asset.get("file_size_bytes")
#         adjustment_signature = asset.get("adjustment_signature")

#         if original_filename is None and filename is None:
#             reason_counter["missing filename and original_filename"] += 1

#         if date is None:
#             reason_counter["missing date"] += 1

#         if path is None:
#             reason_counter["path is None"] += 1
#         elif not os.path.exists(path):
#             reason_counter["path does not exist"] += 1

#         if file_size_bytes is None:
#             reason_counter["file_size_bytes is None"] += 1

#         if adjustment_signature is None:
#             reason_counter["adjustment_signature is None"] += 1

#     print("reason counter:")
#     for reason, count in reason_counter.most_common():
#         print(f"  {reason}: {count}")

#     print()
#     print("missing asset details:")
#     print("-" * 120)

#     for index, asset in enumerate(missing_assets[:max_print], start=1):
#         path = asset.get("path")

#         print(f"{index:02d}.")
#         print("  uuid:", asset.get("uuid"))
#         print("  original_filename:", asset.get("original_filename"))
#         print("  filename:", asset.get("filename"))
#         print("  date:", asset.get("date"))
#         print("  date_added:", asset.get("date_added"))
#         print("  path:", path)
#         print("  path_exists:", None if path is None else os.path.exists(path))
#         print("  file_size_bytes:", asset.get("file_size_bytes"))
#         print("  adjustment_signature:", asset.get("adjustment_signature"))
#         print("  is_movie:", asset.get("is_movie"))
#         print("  hasadjustments:", asset.get("hasadjustments"))
#         print("  path_edited:", asset.get("path_edited"))
#         print("  asset_scope:", asset.get("asset_scope"))
#         print("  albums:", list((asset.get("albums") or {}).values()))
#         print("  folders:", list((asset.get("folders") or {}).values()))
#         print("-" * 120)

# debug_dump_assets_without_photo_library_asset_unique_id(
#     inventory_current,
#     "CURRENT DEFAULT assets without photo_library_asset_unique_id",
# )